# **Analysis of the first half's influence on the final result**

This project is based on the analysis of data provided by StatsBomb relating to the Italian Serie A 2015/16 season. The starting data is organized into two main datasets:
- A dataset (**df_events**) containing event data for each match, which records every game action with related information (event type, coordinates, player, etc.)
- A dataset (**df_agg**) with general information about matches such as final results, time, matchday, etc.

Starting from this data, the project develops through the creation of various analysis tables:
- A *classifica* table that, in addition to points and basic statistics, is enriched with advanced metrics such as Expected Goals, PPDA (Passes per Defensive Action), ball possession, and Field Tilt
- A *match_results* table that aggregates specific statistics from the first half and the complete match for each game.

## Project Structure
The work develops through three main phases:
* **Descriptive analysis** represents the phase where we'll explore the various tables created through different visualizations that will allow us to understand the main characteristics of the teams, as well as the distribution of values in the dataset. Through the use of various metrics and graphical representations, we'll try to identify patterns and trends that characterize the game, paying particular attention to the comparison between first and second half.
* The **feature engineering** phase is dedicated to processing raw data to extract meaningful information. We focus particularly on first-half statistics, creating aggregate metrics for each team. These are then compared with statistics from the complete match to begin understanding how indicative the first half is of the general match trend.
* Finally, in the **modeling** phase, we develop two distinct classification models. The first uses exclusively first-half data to predict the final result, while the second considers the entire match. The comparison between the performances of these models, together with the analysis of the most relevant features, will allow us to quantify how predictive the first half actually is of the final result.

## Research question
This project aims to answer a fundamental question:
"**To what extent are the statistics of the first half of a football match predictive of the final result, and which factors have greater predictive influence?**"

To answer this question, our analysis is articulated in three key aspects:
- Verification of the **independence hypothesis**: we statistically test the null hypothesis that the first half is independent of the final result, using the chi-square test and Cramer's V coefficient.
- **Comparative analysis** of models: we compare the performance of predictive models based only on the first half with those based on the entire match, quantifying the predictive improvement.
- **Feature importance**: we delve into which specific first-half statistics are most influential, analyzing both complete models and models that exclude statistics directly related to goals, to discover more subtle tactical and behavioral patterns.

Our analysis is based on data from 380 matches, providing a significant statistical basis for understanding game dynamics and the importance of the first half in determining the final result.

# Initial libraries and configurations

For this project, we use several fundamental Python libraries for data analysis:

* **pandas** (`import pandas as pd`) is the main library for data manipulation and analysis, which provides us with optimized data structures like DataFrame and Series with functionality for aggregation and data transformation.

* **numpy** (`import numpy as np`) is the fundamental library for numerical computing in Python, which provides support for multi-dimensional arrays and mathematical operations.

* **google.colab.drive** allows us to mount Google Drive in Google Colab, necessary for accessing data files and images saved on Drive.

**Pandas configurations**
```python
pd.set_option('display.max_columns', None)
```
This configuration is particularly useful in our case given the high number of columns in the datasets we will analyze, allowing us to view all details during the exploratory data phase.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', 50)


# Configuration of the visualization environment

For creating our visualizations, we use several Python libraries specialized in plotting and managing sports graphics. In particular, **mplsoccer** provides us with specific functionality for visualizing football data, while **matplotlib** serves as the foundation for all our visualizations.

To maintain a consistent style across all charts, we define a series of elements:
- A set of custom fonts from the Teko family (**Regular**, **Medium**, and **SemiBold**)
- A custom colormap ranging from *blanchedalmond* to *darkred*
- Specific configurations for football pitches through the Pitch class from mplsoccer

In [ ]:
!pip install highlight_text
!pip install mplsoccer

import matplotlib.pyplot as plt
from mplsoccer import Pitch, VerticalPitch
import matplotlib.ticker as ticker
import matplotlib.patheffects as path_effects
import matplotlib.font_manager as fm
import matplotlib.colors as mcolors
from matplotlib import cm
from highlight_text import fig_text, ax_text
from matplotlib.colors import LinearSegmentedColormap, NoNorm
from matplotlib import cm
import matplotlib.gridspec as gridspec
import matplotlib.image as image
import matplotlib.font_manager as font_manager
from matplotlib.patches import RegularPolygon
import matplotlib.gridspec as gridspec
import seaborn as sns

In [ ]:
font_path_regular = '/content/drive/MyDrive/Teko/static/Teko-Regular.ttf'
font_normal = font_manager.FontProperties(fname=font_path_regular)

font_path_med = '/content/drive/MyDrive/Teko/static/Teko-Medium.ttf'
font_med = font_manager.FontProperties(fname=font_path_med)

font_path_semi = '/content/drive/MyDrive/Teko/static/Teko-SemiBold.ttf'
font_semi = font_manager.FontProperties(fname=font_path_semi)

In [ ]:
colors = [
    'blanchedalmond',
    '#ffdcc2',
    '#ffd1ae',
    '#ffc59a',
    '#ffb986',
    '#ffad72',
    '#ffa15e',
    '#ff954a',
    '#ff8936',
    '#ff7d22',
    '#ff710e',
    '#ff6500',
    '#f85800',
    '#f14b00',
    '#ea3e00',
    '#e23100',
    '#d92400',
    '#c71700',
    '#a31212',
    '#8B0000'   # darkred
]

soc_cm = LinearSegmentedColormap.from_list('SOC4', colors, N=50)
plt.colormaps.register(cmap=soc_cm)

## Importing the two main datasets

Let's begin by exploring the dataset that contains all events from the Serie A 2015/16 matches.

In [ ]:
drive.mount('/content/drive')

file_path = '/content/drive/My Drive/seriea1516.csv'

df_events = pd.read_csv(file_path)
df_events.head()

In [ ]:
df_events.shape

The dataset contains 1,353,739 rows and 176 columns, where each row represents a match event (a pass, a shot, a defensive action, etc.) and the columns contain the various characteristics of the event.


In [ ]:
df_events.info()

From this overview, we can see the types of data present in the dataset and identify any missing values. Many columns are of type 'object' or float64, which is typical for event data that combines categorical information (event type, player name) and numerical (coordinates, statistics).

In [ ]:
list(df_events.columns)

The dataset has a complex and detailed structure, with columns that can be organized into different categories:

**Event and match identification**
- Basic columns like *id*, *index*, *match_id* to uniquely identify events and matches
- References to competition and season to contextualize the data

**Temporal and sequence data**
- A complete system to track when each event occurs:
 - *period*: first or second half
 - *minute*, *second*: precise moment of the event
 - *timestamp*: complete timestamp
 - *ElapsedTime*: time elapsed from the beginning
 - *TimeInPoss*: duration of possession

**Position and movement**
- Coordinate system to map events on the field:
 - *location.x/y*: where the event begins
 - *carry.end_location.x/y*: where a ball movement ends
 - *pass.end_location.x/y*: where a pass arrives
 - *shot.end_location.x/y/z*: complete trajectory of shots

**Event qualifiers**
- Each event type (**Pass**, *Shot*, *Duel*, etc.) has its own set of qualifying columns:
 - For passes: height, type, outcome, body part
 - For shots: xG, body part, technique, outcome
 - For duels: type of duel, outcome
 - For saves: type, technique, outcome

**Contextual and tactical metrics**
- *under_pressure*: indicates if the action is under pressure
- *counterpress*: identifies counterpressing actions
- *possession*: tracks possession sequences
- *tactics.formation*: tactical formation
- Various distance and angle metrics for more detailed analysis

**Player and team reference columns**
- Identifiers and names of:
 - Players involved
 - Teams
 - Roles
 - Recipient players (for passes)

In [ ]:
df_events["type.name"].unique()

We have examined the unique values in the *type.name* column because it's fundamental to understand which types of events are tracked in the dataset.

The events can be grouped into different categories:

**Ball possession events**:
- Pass, Carry, Ball Receipt: represent the movement of the ball
- Shot: shooting actions, fundamental for our analysis of results
- Dribble: individual actions that can create superiority

**Defensive events**:
- Pressure, Clearance, Block, Interception: different types of defensive actions
- Ball Recovery: ball recoveries that can start new actions
- Duel: direct confrontations between players, both aerial and on the ground

**Tactical/administrative events**:
- Starting XI, Substitution: team management
- Tactical Shift: changes in strategy
- Half Start/End: define the periods of play

This richness of events will allow us to build a detailed analysis not only of the results but also of the teams' playing styles.

In [ ]:
# Data Preparation
event_counts = df_events['type.name'].value_counts()

# Create figure
fig, ax = plt.subplots(figsize=(20, 12), facecolor='blanchedalmond')
plt.gca().set_facecolor('blanchedalmond')

# Create barplot
bars = ax.bar(event_counts.index, event_counts.values, color='darkred', alpha=0.6)

# Title and description
plt.title('Event Type Distribution', x=-0.04, fontproperties=font_semi,
 fontsize=40, color="darkred", pad=50, loc='left')
fig.text(0.005, 0.92, "Pass-related events are the most frequent",
 fontproperties=font_semi, fontsize=30, color="darkorange")

# Axis labels
plt.xticks(rotation=45, ha='right', fontproperties=font_normal, fontsize=18, color='black')
plt.yticks(fontproperties=font_normal, fontsize=18, color='black')

# Axis styling
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.grid(True, color="white", linestyle='--', linewidth=0.5)

# Add values above bars
for i, v in enumerate(event_counts.values):
 ax.text(i, v, f'{v:,.0f}', ha='center', va='bottom',
 fontproperties=font_normal, fontsize=14, color='black')

# Add StatsBomb logo
logo = plt.imread('/content/drive/MyDrive/SB - Icon Lockup - Colour positive.png')
logo_ax = fig.add_axes([0.85, -0.02, 0.1, 0.02])
logo_ax.imshow(logo)
logo_ax.axis('off')
plt.tight_layout()
plt.show()

Let's now analyze the dataset that contains the general information of each match in Serie A 2015/16.

In [ ]:
df_agg = pd.read_csv('/content/drive/My Drive/aggregate_ita.csv')
df_agg.head()

In [ ]:
df_agg.shape

The dataset contains 380 rows (one for each match in the championship) and 41 columns that describe various aspects of each match.

In [ ]:
df_agg.info()

This dataset contains general information on each match.

**Key information for the analysis**:
- *match_id*: unique identifier of the match
- *home_score*, *away_score*: the final results, fundamental for our analysis
- *home_team.home_team_name*, *away_team.away_team_name*: team names

**Contextual information**:
- *match_week*: championship matchday
- *match_date*, *kick_off*: date and time of the match

**Other information**:
- Details on the competition and season
- Information on the stadium
- Data on the referee
- Various metadata on data quality

Our attention will focus mainly on the final results (home_score and away_score), which we will use to create our match_results table and validate the statistics calculated from the events dataset.

# Creation of the *classifica* table and advanced metrics

The standings table is progressively enriched with various aggregate statistics starting from basic information such as points, wins, draws, losses, and goals. To these are added more detailed statistics like total shots, shots on target, and passing accuracy.

Particular attention is given to some advanced metrics that provide us with deeper information on the style and effectiveness of the teams' play:

* **Expected Goals (xG)**: this metric evaluates the probability that a shot will turn into a goal based on various factors such as the shot position, shooting angle, and game situation. The comparison between xG and actual goals allows us to evaluate the quality of created chances and the finishing efficiency of teams.

* **Passes per Defensive Action (PPDA)**: this is an indicator of a team's pressing intensity. It is calculated by dividing the number of passes conceded to opponents in their own half by the number of defensive actions (tackles, interceptions, fouls) made in the same area. A lower value indicates more intense pressing.

* **Field Tilt**: this metric measures a team's territorial dominance, calculated as the percentage of passes made in the offensive third of the field relative to the total passes made in that area by both teams. A higher value indicates a greater ability to play in the opponent's half.

* **Key passes**: these represent passes that directly lead to a shot on goal, highlighting a team's ability to create goal-scoring opportunities. A high number of key passes indicates an excellent offensive build-up capability and final third execution.

In [ ]:
def create_standings(df):
    """
    Create a standings table from match results.

    Parameters:
    df (DataFrame): DataFrame containing match data with columns for team names and scores.

    Returns:
    DataFrame: A sorted standings table with team statistics.
    """
    # Dictionary that will contain team names as keys and their statistics as values
    standings = {}

    for _, match in df.iterrows():  # Iterate over each row where match is the current row. The * ignores the row index.
        # Extract match data
        home_team = match['home_team.home_team_name']
        away_team = match['away_team.away_team_name']
        home_goals = match['home_score']
        away_goals = match['away_score']

        # Initialize teams if not present with all values at zero
        for team in [home_team, away_team]:
            if team not in standings:
                standings[team] = {
                    'points': 0,
                    'played': 0,
                    'won': 0,
                    'drawn': 0,
                    'lost': 0,
                    'goals_for': 0,
                    'goals_against': 0,
                    'goal_difference': 0
                }

        # Update home team statistics
        standings[home_team]['played'] += 1
        standings[home_team]['goals_for'] += home_goals
        standings[home_team]['goals_against'] += away_goals

        # Update away team statistics
        standings[away_team]['played'] += 1
        standings[away_team]['goals_for'] += away_goals
        standings[away_team]['goals_against'] += home_goals

        # Assign points
        if home_goals > away_goals:  # Home win
            standings[home_team]['points'] += 3
            standings[home_team]['won'] += 1
            standings[away_team]['lost'] += 1
        elif home_goals < away_goals:  # Away win
            standings[away_team]['points'] += 3
            standings[away_team]['won'] += 1
            standings[home_team]['lost'] += 1
        else:  # Draw
            standings[home_team]['points'] += 1
            standings[away_team]['points'] += 1
            standings[home_team]['drawn'] += 1
            standings[away_team]['drawn'] += 1

        # Update goal difference
        for team in [home_team, away_team]:
            standings[team]['goal_difference'] = standings[team]['goals_for'] - standings[team]['goals_against']

    # Convert dictionary to DataFrame
    standings_df = pd.DataFrame.from_dict(standings, orient='index')  # keys become indices

    # Sort by points, goal difference, and goals scored
    standings_df = standings_df.sort_values(['points', 'goal_difference', 'goals_for'],
                                           ascending=[False, False, False])

    return standings_df

# Create standings table from aggregate match data
classifica = create_standings(df_agg)

In [ ]:
def calculate_possession_by_passes(events_df):
    """
    Calculate possession percentage for each team based on pass counts.

    Parameters:
    events_df (DataFrame): DataFrame containing event data with pass information.

    Returns:
    Series: Average possession percentage for each team across all matches.
    """
    # Filter only "Pass" events, group by "match_id" and "team_name",
    # convert to df and assign the name "passes" to the column containing counts
    passes_by_match = events_df[events_df['type.name'] == 'Pass'].groupby(
        ['match_id', 'team.name']
    ).size().reset_index(name='passes')

    # Empty list for dictionaries with possession data for each team in each match
    possession_stats = []

    # Iterate over each unique ID in the passes DataFrame
    for match_id in passes_by_match['match_id'].unique():
        # Create a subset containing only passes for the current match
        match_passes = passes_by_match[passes_by_match['match_id'] == match_id]

        # Sum all passes made
        total_passes = match_passes['passes'].sum()

        # Iterate over each row of the current match DataFrame (each row represents a team)
        for _, row in match_passes.iterrows():
            possession_pct = (row['passes'] / total_passes) * 100
            possession_stats.append({
                'team': row['team.name'],
                'possession_pct': possession_pct
            })

    # Calculate the average possession for each team
    possession_df = pd.DataFrame(possession_stats)
    avg_possession = possession_df.groupby('team')['possession_pct'].mean().round(2)

    return avg_possession

# Calculate possession statistics from events data
possession_stats = calculate_possession_by_passes(df_events)

# Add possession data to the standings table
classifica = classifica.merge(
    possession_stats.to_frame('possession'),
    left_index=True,
    right_index=True,
    how='left'
)


In [ ]:
def calculate_field_tilt(events_df):
    """
    Calculate Field Tilt for each team.

    Field Tilt is defined as:
    (team passes in final third) / (all passes in final third) * 100

    Parameters:
    events_df (DataFrame): DataFrame containing event data with pass and location information.

    Returns:
    DataFrame: Field Tilt percentage for each team.
    """
    field_tilt_stats = {}  # Dictionary to store values for each team

    # Iterate over each unique team name in events_df
    for team in events_df['team.name'].unique():
        total_passes_final_third = 0
        team_passes_final_third = 0

        # Find all matches involving the team
        team_matches = events_df[
            (events_df['team.name'] == team) |
            (events_df['possession_team.name'] == team)
        ]['match_id'].unique()

        # Iterate over each match ID found in the previous step
        for match_id in team_matches:
            # Create a subset containing only events from the current match
            match_df = events_df[events_df['match_id'] == match_id]

            # Team passes in the final third
            team_passes = len(match_df[
                (match_df['type.name'] == 'Pass') &
                (match_df['team.name'] == team) &
                (match_df['location.x'] > 80)  # final third
            ])

            # All passes in the final third in this match
            all_passes = len(match_df[
                (match_df['type.name'] == 'Pass') &
                (match_df['location.x'] > 80)
            ])

            team_passes_final_third += team_passes
            total_passes_final_third += all_passes

        # Calculate field tilt
        if total_passes_final_third > 0:
            field_tilt = (team_passes_final_third / total_passes_final_third) * 100
            field_tilt_stats[team] = round(field_tilt, 2)
        else:
            field_tilt_stats[team] = None

    return pd.DataFrame.from_dict(field_tilt_stats, orient='index', columns=['field_tilt'])

# Calculate field tilt statistics from events data
field_tilt_stats = calculate_field_tilt(df_events)

# Add field tilt data to the standings table
classifica = classifica.merge(
    field_tilt_stats,
    left_index=True,
    right_index=True,
    how='left'
)

In [ ]:
def calculate_team_xg_and_goals_diff(events_df, standings_df):
    """
    Calculate Expected Goals (xG) statistics and compare with actual goals for each team.

    Parameters:
    events_df (DataFrame): DataFrame containing event data with shot and xG information.
    standings_df (DataFrame): Current standings table to merge with xG statistics.

    Returns:
    DataFrame: Updated standings with xG statistics and goal-xG differentials.
    """
    # Filter only shots with xG, excluding nan values
    shots_df = events_df[~events_df['shot.statsbomb_xg'].isna()]

    # Accumulated xG for each team
    team_xg_for = shots_df.groupby('team.name')['shot.statsbomb_xg'].sum()

    # xG conceded
    team_xg_against = {}
    for team in shots_df['team.name'].unique():
        # For each match where this team played, keep the IDs
        team_matches = shots_df[
            (shots_df['team.name'] == team) |
            (shots_df['possession_team.name'] == team)
        ]['match_id'].unique()

        # Sum of opponents' xG in these matches
        xg_against = shots_df[
            (shots_df['match_id'].isin(team_matches)) &  # only this team's matches
            (shots_df['team.name'] != team)  # only opponents' shots
        ]['shot.statsbomb_xg'].sum()

        team_xg_against[team] = xg_against

    # Create DataFrame with xG statistics
    team_stats = pd.DataFrame({
        'team_name': team_xg_for.index,
        'total_xG_for': team_xg_for.values.round(2),
        'total_xG_against': [team_xg_against[team] for team in team_xg_for.index]
    })

    # Merge with standings and calculate differences between actual goals and xG
    final_stats = standings_df.merge(
        team_stats,
        left_index=True,
        right_on='team_name',
        how='left'
    ).set_index('team_name')

    final_stats['xG_goals_diff'] = (final_stats['total_xG_for'] - final_stats['goals_for']).round(2)
    final_stats['xG_conceded_diff'] = (final_stats['total_xG_against'] - final_stats['goals_against']).round(2)

    return final_stats

# Calculate xG statistics and merge with standings
classifica = calculate_team_xg_and_goals_diff(df_events, classifica)


In [ ]:
def calculate_shot_stats(events_df):
    """
    Calculate shot statistics for each team:
    - Total shots
    - Shots on target (Goal + Saved + Saved to Post)

    Parameters:
    events_df (DataFrame): DataFrame containing event data with shot information.

    Returns:
    DataFrame: Shot statistics for each team.
    """
    shot_stats = {}

    # For each team
    for team in events_df['team.name'].unique():
        # Filter all shots by the team
        team_shots = events_df[
            (events_df['type.name'] == 'Shot') &
            (events_df['team.name'] == team)
        ]

        # Total shots
        total_shots = len(team_shots)

        # Shots on target (Goal, Saved, Saved to Post)
        shots_on_target = len(team_shots[
            team_shots['shot.outcome.name'].isin(['Goal', 'Saved', 'Saved to Post'])
        ])

        shot_stats[team] = {
            'total_shots': total_shots,
            'shots_on_target': shots_on_target
        }

    return pd.DataFrame.from_dict(shot_stats, orient='index')

# Calculate shot statistics from events data
shot_stats = calculate_shot_stats(df_events)

# Add shot statistics to the standings table
classifica = classifica.merge(
    shot_stats,
    left_index=True,
    right_index=True,
    how='left'
)

In [ ]:
def calculate_pass_stats(events_df):
    """
    Calculate passing statistics for each team, divided by thirds of the field.

    Parameters:
    events_df (DataFrame): DataFrame containing event data with pass information.

    Returns:
    DataFrame: Passing statistics for each team by field zone.
    """
    pass_stats = {}

    # Define the thirds of the field
    defensive_third = 40
    middle_third = 80

    for team in events_df['team.name'].unique():
        # Filter passes by the team
        team_passes = events_df[
            (events_df['type.name'] == 'Pass') &
            (events_df['team.name'] == team)
        ]

        # Dictionary for this team
        team_stats = {}

        # Dictionary of boolean masks to filter passes based on field position
        zones = {
            'defensive': team_passes['location.x'] <= defensive_third,
            'middle': (team_passes['location.x'] > defensive_third) & (team_passes['location.x'] <= middle_third),
            'attacking': team_passes['location.x'] > middle_third
        }

        # Total statistics
        total_attempted = len(team_passes)
        total_completed = len(team_passes[team_passes['pass.outcome.name'].isna()])
        completion_pct = round((total_completed / total_attempted * 100), 2) if total_attempted > 0 else 0

        team_stats.update({
            'total_passes_attempted': total_attempted,
            'total_passes_completed': total_completed,
            'total_passes_pct': completion_pct
        })

        # For each field zone
        for zone_name, zone_mask in zones.items():
            # Filter team passes to include only those in current zone
            zone_passes = team_passes[zone_mask]
            attempted = len(zone_passes)
            completed = len(zone_passes[zone_passes['pass.outcome.name'].isna()])
            completion_pct = round((completed / attempted * 100), 2) if attempted > 0 else 0

            team_stats.update({
                f'{zone_name}_passes_attempted': attempted,
                f'{zone_name}_passes_completed': completed,
                f'{zone_name}_passes_pct': completion_pct
            })

        pass_stats[team] = team_stats

    return pd.DataFrame.from_dict(pass_stats, orient='index')

# Calculate passing statistics from events data
pass_stats = calculate_pass_stats(df_events)

# Add passing statistics to the standings table
classifica = classifica.merge(
    pass_stats,
    left_index=True,
    right_index=True,
    how='left'
)


In [ ]:
def calculate_additional_defensive_stats(events_df):
    """
    Calculate additional defensive statistics:
    - Successful tackles + interceptions
    - Fouls committed
    - Yellow cards
    - Red cards

    Parameters:
    events_df (DataFrame): DataFrame containing event data with defensive actions.

    Returns:
    DataFrame: Defensive statistics for each team.
    """
    stats = {}

    for team in events_df['team.name'].unique():
        # Successful tackles (from Duels)
        successful_tackles = len(events_df[
            (events_df['type.name'] == 'Duel') &
            (events_df['team.name'] == team) &
            (events_df['duel.type.name'] == 'Tackle') &
            (events_df['duel.outcome.name'].isin(['Success In Play', 'Won', 'Success Out']))
        ])

        # Successful interceptions
        successful_interceptions = len(events_df[
            (events_df['type.name'] == 'Interception') &
            (events_df['team.name'] == team) &
            (events_df['interception.outcome.name'].isin(['Won', 'Success In Play', 'Success Out']))
        ])

        # Tackles + Interceptions
        tkl_int = successful_tackles + successful_interceptions

        # Fouls committed
        fouls = len(events_df[
            (events_df['type.name'] == 'Foul Committed') &
            (events_df['team.name'] == team)
        ])

        # Yellow and red cards
        yellow_cards = len(events_df[
            (events_df['type.name'] == 'Foul Committed') &
            (events_df['team.name'] == team) &
            (events_df['foul_committed.card.name'] == 'Yellow Card')
        ])

        red_cards = len(events_df[
            (events_df['type.name'] == 'Foul Committed') &
            (events_df['team.name'] == team) &
            (events_df['foul_committed.card.name'] == 'Red Card')
        ])

        stats[team] = {
            'tkl_int': tkl_int,
            'fouls': fouls,
            'yellow_cards': yellow_cards,
            'red_cards': red_cards
        }

    return pd.DataFrame.from_dict(stats, orient='index')

# Calculate additional defensive statistics from events data
additional_stats = calculate_additional_defensive_stats(df_events)

# Add defensive statistics to the standings table
classifica = classifica.merge(
    additional_stats,
    left_index=True,
    right_index=True,
    how='left'
)

In [ ]:
def calculate_ppda(events_df):
    """
    Calculate PPDA (Passes per Defensive Action) for each team.
    A lower PPDA indicates more intense pressing.

    Parameters:
    events_df (DataFrame): DataFrame containing event data.

    Returns:
    DataFrame: PPDA statistics for each team.
    """
    ppda_stats = {}
    midfield = 60

    for team in events_df['team.name'].unique():
        total_opp_passes = 0
        total_def_actions = 0

        # Find all matches involving the team
        team_matches = events_df[
            (events_df['team.name'] == team) |
            (events_df['possession_team.name'] == team)
        ]['match_id'].unique()

        for match_id in team_matches:
            match_df = events_df[events_df['match_id'] == match_id]

            # Successful passes by opponents in their half
            opp_passes = len(match_df[
                (match_df['type.name'] == 'Pass') &
                (match_df['team.name'] != team) &
                (match_df['location.x'] < midfield) &
                (match_df['pass.outcome.name'].isna())  # successful passes
            ])

            # Defensive actions in the same zone
            # Defensive actions in our offensive half
            def_actions = len(match_df[
                (match_df['team.name'] == team) &
                (match_df['location.x'] > midfield) &
                (
                    # Successful tackles
                    (
                        (match_df['type.name'] == 'Duel') &
                        (match_df['duel.type.name'] == 'Tackle') &
                        (match_df['duel.outcome.name'].isin(['Success In Play', 'Won', 'Success Out']))
                    ) |
                    # Successful interceptions
                    (
                        (match_df['type.name'] == 'Interception') &
                        (match_df['interception.outcome.name'].isin(['Won', 'Success In Play', 'Success Out']))
                    ) |
                    # Block (excluding goalkeeper save_block)
                    (
                        (match_df['type.name'] == 'Block') &
                        (match_df['block.save_block'].isna())
                    ) |
                    # Clearance (all)
                    (match_df['type.name'] == 'Clearance') |
                    # Won 50/50
                    (
                        (match_df['type.name'] == '50/50') &
                        (match_df['50_50.outcome.name'].isin(['Won', 'Success To Team']))
                    ) |
                    # Fouls committed in offensive phase
                    (
                        (match_df['type.name'] == 'Foul Committed')
                    )
                )
            ])

            total_opp_passes += opp_passes
            total_def_actions += def_actions

        # Final PPDA calculation
        if total_def_actions > 0:
            ppda_stats[team] = round(total_opp_passes / total_def_actions, 2)
        else:
            ppda_stats[team] = None

    return pd.DataFrame.from_dict(ppda_stats, orient='index', columns=['ppda'])

# Calculate PPDA statistics from events data
ppda_stats = calculate_ppda(df_events)

# Add PPDA statistics to the standings table
classifica = classifica.merge(
    ppda_stats,
    left_index=True,
    right_index=True,
    how='left'
)

In [ ]:
def calculate_opponents_pass_accuracy(events_df):
   """
   Calculate opponents' pass accuracy against each team.

   Parameters:
   events_df (DataFrame): DataFrame containing event data.

   Returns:
   DataFrame: Opponents' pass accuracy statistics for each team.
   """
   pass_accuracy_against = {}

   for team in events_df['team.name'].unique():
       total_passes = 0
       successful_passes = 0

       # Find all matches involving the team
       team_matches = events_df[
           (events_df['team.name'] == team) |
           (events_df['possession_team.name'] == team)
       ]['match_id'].unique()

       for match_id in team_matches:
           match_df = events_df[events_df['match_id'] == match_id]

           # All passes by opponents
           opp_passes = match_df[
               (match_df['type.name'] == 'Pass') &
               (match_df['team.name'] != team)
           ]

           # Count total and successful passes
           total_passes += len(opp_passes)
           successful_passes += len(opp_passes[opp_passes['pass.outcome.name'].isna()])

       # Calculate average accuracy
       if total_passes > 0:
           accuracy = (successful_passes / total_passes) * 100
           pass_accuracy_against[team] = round(accuracy, 2)
       else:
           pass_accuracy_against[team] = None

   return pd.DataFrame.from_dict(pass_accuracy_against, orient='index', columns=['opp_pass_accuracy'])

# Calculate opponents' pass accuracy statistics from events data
pass_accuracy_stats = calculate_opponents_pass_accuracy(df_events)

# Add opponents' pass accuracy statistics to the standings table
classifica = classifica.merge(
   pass_accuracy_stats,
   left_index=True,
   right_index=True,
   how='left'
)

In [ ]:
def calculate_key_passes(events_df):
   """
   Calculate the number of key passes for each team.
   Key passes are passes that lead directly to a shot.

   Parameters:
   events_df (DataFrame): DataFrame containing event data with pass information.

   Returns:
   DataFrame: Key passes statistics for each team.
   """
   key_passes = events_df[
       (events_df['type.name'] == 'Pass') &
       ((events_df['pass.shot_assist'] == True) |
        (events_df['pass.goal_assist'] == True) |
        (~events_df['pass.assisted_shot_id'].isna()))
   ].groupby('team.name').size()

   return pd.DataFrame(key_passes).rename(columns={0: 'key_passes'})

# Calculate key passes statistics from events data
key_passes_stats = calculate_key_passes(df_events)

# Add key passes statistics to the standings table
classifica = classifica.merge(
   key_passes_stats,
   left_index=True,
   right_index=True,
   how='left'
)

In [ ]:
def calculate_pressure_height(events_df):
    """
    Calculate the median height of pressure for each team.
    A higher value indicates pressing higher up the pitch.

    Parameters:
    events_df (DataFrame): DataFrame containing event data with pressure information.

    Returns:
    DataFrame: Pressure height statistics for each team.
    """
    # Filter pressure events
    pressure_events = events_df[events_df['type.name'] == 'Pressure']

    # Calculate the median
    pressure_height = pressure_events.groupby('team.name')['location.x'].median()
    pressure_height_df = pressure_height.to_frame('pressure_height')

    return pressure_height_df

# Calculate pressure height statistics from events data
pressure_stats = calculate_pressure_height(df_events)

# Add pressure height statistics to the standings table
classifica = classifica.merge(
    pressure_stats,
    left_index=True,
    right_index=True,
    how='left'
)

In [ ]:
def calculate_progressive_passes(events_df):
   """
   Calculate the total progressive passes for each team.
   A pass is considered progressive if it reduces the distance to the goal by at least 20%.

   Parameters:
   events_df (DataFrame): DataFrame containing event data with pass information.

   Returns:
   DataFrame: Progressive passes statistics for each team.
   """
   # Filter completed passes
   completed_passes = events_df[
       (events_df['type.name'] == 'Pass') &
       (events_df['pass.outcome.name'].isna())
   ]

   # Function to determine if a pass is progressive
   def is_progressive(row):
       start_dist = np.sqrt((120 - row['location.x'])**2 + (40 - row['location.y'])**2)
       end_dist = np.sqrt((120 - row['pass.end_location.x'])**2 + (40 - row['pass.end_location.y'])**2)
       return end_dist < start_dist * 0.8  # at least 20% reduction

   # Count progressive passes for each team
   progressive_passes = completed_passes[completed_passes.apply(is_progressive, axis=1)]
   prog_passes_count = progressive_passes.groupby('team.name').size()
   prog_passes_df = pd.DataFrame(prog_passes_count, columns=['progressive_passes'])

   return prog_passes_df

# Calculate progressive passes statistics from events data
prog_passes_stats = calculate_progressive_passes(df_events)

# Add progressive passes statistics to the standings table
classifica = classifica.merge(
   prog_passes_stats,
   left_index=True,
   right_index=True,
   how='left'
)

In [ ]:
classifica.columns

In [ ]:
classifica

### Point Distribution and Correlation

The 2015/16 season shows a particularly interesting distribution of points:
- **Dispersion**: With an average of 52.25 points and a standard deviation of 17.25, we observe a considerable variability in performance. The coefficient of variation of 33% confirms a championship characterized by strong disparities.
- **Championship structure**: The quartile analysis reveals a clear stratification:
  * First quartile: 39.75 points
  * Median: 46 points
  * Third quartile: 61.75 points
  * IQR: 22 points
  
  This interquartile range of 22 points suggests a significant dispersion in the central area of the standings, indicating a very competitive championship in the middle range.
- **Asymmetry**: The skewness value of 0.8 indicates a slightly positive asymmetric distribution. This means that while most teams concentrated in the mid-low range, some significantly overperformed, "pulling" the distribution to the right.

### Correlation Analysis

The correlations between the different metrics reveal significant patterns:
1. Scores ("points") show very strong correlations with offensive metrics like "goals_for" (0.91) and also show strong connections with game control indicators such as "possession" (0.83), "field_tilt" (0.84), and "key_passes" (0.85). This suggests that teams that accumulate more points tend to dominate ball possession and create more goal-scoring opportunities.
2. Interestingly, "progressive_passes" and "pressure_height" are strongly correlated with each other (0.90), indicating that teams that practice high pressing also tend to make more progressive passes.
3. The "ppda" and "tkl_int" metrics seem to have more moderate correlations with other parameters, suggesting that they represent more independent aspects of the game.

In [ ]:
# Calculate descriptive statistics for selected columns
stats_desc = classifica[['points', 'goals_for', 'goals_against', 'possession', "fouls", "total_passes_pct",
 'total_xG_for', 'total_xG_against', 'ppda', "tkl_int", "key_passes", "pressure_height", "progressive_passes"]].describe()

# Calculate correlations between selected metrics
correlations = classifica[['points', 'goals_for', 'goals_against', 'possession', "fouls", "total_passes_pct",
'field_tilt', 'total_xG_for', 'ppda', "tkl_int", "key_passes", "pressure_height", "progressive_passes"]].corr()

# Descriptive statistics
print("\nDescriptive Statistics of Points:")
print(classifica['points'].describe())

# Quartiles and IQR
quartiles = classifica['points'].quantile([0.25, 0.5, 0.75])
iqr = quartiles[0.75] - quartiles[0.25]
print("\nQuartile Analysis:")
print(f"First quartile (25%): {quartiles[0.25]:.1f}")
print(f"Median (50%): {quartiles[0.5]:.1f}")
print(f"Third quartile (75%): {quartiles[0.75]:.1f}")
print(f"IQR: {iqr:.1f}")

# Skewness and CV
skewness = classifica['points'].skew()
cv = (classifica['points'].std() / classifica['points'].mean()) * 100
print(f"\nSkewness: {skewness:.3f}")
print(f"\nCoefficient of Variation: {cv:.1f}%")

# Create figure with two subplots
fig = plt.figure(figsize=(20, 7), dpi=300, facecolor='blanchedalmond')
gs = fig.add_gridspec(1, 2, hspace=0.3, wspace=0.3)

# 1. Points distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor('blanchedalmond')

# Create boxplot
sns.boxplot(x=classifica['points'], ax=ax1, color='darkred', width=0.5, linewidth=2)

# Add actual points as overlay
sns.stripplot(x=classifica['points'], ax=ax1, color='black', alpha=0.5, size=10)

ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.set_title('POINTS DISTRIBUTION', fontproperties=font_semi, color='darkred', size=14, pad=20)
ax1.set_xlabel('Points', fontproperties=font_med, color='black', size=13)
ax1.grid(True, linestyle='--', alpha=0.3, color='black')
ax1.set_axisbelow(True)
ax1.set_xticklabels(ax1.get_xticks(), fontproperties=font_med, size=12)
ax1.set_yticklabels([])  # Hide y-axis labels

# 2. Correlation heatmap
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor('blanchedalmond')

# Keep only the lower half of the matrix
mask = np.zeros_like(correlations, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True
im = ax2.imshow(np.ma.array(correlations, mask=mask), cmap='SOC4', vmin=-1, vmax=1)

# Add values in the matrix
for i in range(len(correlations.columns)):
    for j in range(len(correlations.index)):
        if not mask[i, j]:  # Only for the lower part of the matrix
            text = ax2.text(j, i, f'{correlations.iloc[i, j]:.2f}',
                          ha="center", va="center", color="black", fontproperties=font_med, fontsize=12)

# Add labels and colorbar
cbar = fig.colorbar(im, ax=ax2, label='Correlation')
cbar.ax.yaxis.set_tick_params(labelsize=12)
for label in cbar.ax.yaxis.get_ticklabels():
    label.set_fontproperties(font_med)

ax2.set_xticks(np.arange(len(correlations.columns)))
ax2.set_yticks(np.arange(len(correlations.index)))
ax2.set_xticklabels(correlations.columns, fontproperties=font_med, size=12, rotation=45, ha="right")
ax2.set_yticklabels(correlations.index, fontproperties=font_med, size=12)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.set_title('CORRELATIONS BETWEEN METRICS', fontproperties=font_semi, color='darkred', size=14, pad=20)

# StatsBomb Logo
logo = plt.imread('/content/drive/MyDrive/SB - Icon Lockup - Colour positive.png')
logo_ax = fig.add_axes([0.45, -0.05 , 0.1, 0.09])
logo_ax.imshow(logo)
logo_ax.axis('off')

plt.show()

# Data Visualization

After statistically analyzing the composition of the dataset, let's delve into the distribution of statistics by individual teams, contextualizing the results tactically.

## Scoring Efficiency and Finishing
Let's begin by analyzing the relationship between expected goals (xG) and actual goals scored, then examine the performance of the best finishers:
- Comparison between goals scored and xG for each team
- Detailed shot maps of the top 6 scorers in the championship
- Analysis of scoring efficiency through metrics such as xG/shots

## Creativity and Game Construction
The analysis of key passes reveals the main playmakers:
- Mapping of key passes of the 6 best players
- Distinction between key passes and actual assists
- Game creation patterns in different areas of the field

## Playing Styles and Territorial Control
The passing accuracy by field zone shows us how each team builds their play:
- Accuracy maps in the three thirds of the field for each team
- Identification of areas of strength and weakness
- Differences in approach to game construction

## Pressing
We conclude with an analysis of the tactical approach in teams' non-possession:
- Relationship between pressing intensity (PPDA) and its effectiveness and pressing zones

These technical visualizations allow us to go beyond pure numbers, offering an in-depth understanding of the tactical choices and distinctive characteristics of each team in the championship.

## Barplot comparing xG and goals scored

In [ ]:
# Create a copy of the dataframe and reverse the order
classifica_plot = classifica.iloc[::-1].copy()

fig = plt.figure(figsize=(6,3.5), dpi=300, facecolor="blanchedalmond")
ax = plt.subplot(111)
ax.set_facecolor('blanchedalmond')
plt.gca().set_facecolor('blanchedalmond')

# Remove left spine and y-ticks
ax.spines['left'].set_visible(False)
ax.set_yticks([])

# Set x-axis ticks every 5 units
ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
ax.tick_params(labelsize=7, colors="black")
ax.set_xticklabels(np.arange(-5, 90, 5), fontproperties=font_normal, fontsize=7)
ax.grid(axis='x', color='lightgrey', ls=':')

# Expected Goals bars
bars_ = ax.barh(range(len(classifica_plot)), classifica_plot['total_xG_for'], height=0.65)
for bar in bars_:
    bar.set_zorder(1)
    bar.set_facecolor('none')
    x, y = bar.get_xy()  # extract position
    w, h = bar.get_width(), bar.get_height()  # and dimensions
    grad = np.atleast_2d(np.linspace(0, 1*w/max(classifica_plot['total_xG_for']), 256))  # create normalized linear gradient
    ax.imshow(
        grad, extent=[x, x+w, y, y+h],
        aspect='auto', zorder=3,
        norm=NoNorm(vmin=0, vmax=1), cmap='SOC4', alpha=0.45
    )

# Actual Goals bars
bars_ = ax.barh(range(len(classifica_plot)), classifica_plot['goals_for'], height=0.3)
for bar in bars_:
    bar.set_zorder(1)
    bar.set_facecolor('none')
    x, y = bar.get_xy()
    w, h = bar.get_width(), bar.get_height()
    grad = np.atleast_2d(np.linspace(0, 1*w/max(classifica_plot['goals_for']), 256))
    ax.imshow(
        grad, extent=[x, x+w, y, y+h],
        aspect='auto', zorder=3,
        norm=NoNorm(vmin=0, vmax=1), cmap='SOC4'
    )

# Set plot limits
ax.set_xlim(0, max(classifica_plot['total_xG_for'].max(), classifica_plot['goals_for'].max()) + 5)
ax.set_ylim(-.5, len(classifica_plot)-.5)

# Set visible spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color("black")

# Add team names and goal differentials
for i in range(len(classifica_plot)):
    # Add team name
    ax.annotate(
        xy=(0, i),
        text=classifica_plot.index[i],
        size=7,
        c="black",
        ha='right',
        va='center',
        fontproperties=font_med
    )

    # Add goal differential
    diff = classifica_plot['goals_for'].iloc[i] - classifica_plot['total_xG_for'].iloc[i]
    text_sign = '+' if diff > 0 else ''
    text_ = ax.annotate(
        xy=(classifica_plot['goals_for'].iloc[i], i),
        xytext=(8, 0),
        text=f'{text_sign}{diff:.1f}',
        size=6,
        c="black",
        ha='center',
        va='center',
        textcoords='offset points',
        weight='bold',
        fontproperties=font_semi
    )
    text_.set_path_effects(
        [path_effects.Stroke(linewidth=1.5, foreground='white'), path_effects.Normal()]
    )

# Add "Goals" annotation with arrow
text_ = ax.annotate(
    xy=(classifica_plot['goals_for'].iloc[len(classifica_plot)-2], len(classifica_plot)-3),  # third from top
    xytext=(-30, -20),
    text='Goals',
    c="black",
    size=7,
    ha='center',
    va='center',
    textcoords='offset points',
    weight='bold',
    fontproperties=font_semi,
    arrowprops=dict(
        arrowstyle="->", shrinkA=0, shrinkB=5, color="black", linewidth=0.75,
        connectionstyle="angle3,angleA=10,angleB=-100"
    )
)

# Add "Expected Goals" annotation with arrow
text_ = ax.annotate(
    xy=(classifica_plot['total_xG_for'].iloc[4], 4),  # fifth from bottom
    xytext=(60, -20),
    text='Expected Goals',
    c="black",
    size=7,
    ha='center',
    va='center',
    textcoords='offset points',
    weight='bold',
    fontproperties=font_semi,
    arrowprops=dict(
        arrowstyle="->", shrinkA=0, shrinkB=5, color="black", linewidth=0.75,
        connectionstyle="angle3,angleA=10,angleB=-100"
    )
)

# Add StatsBomb logo
logo = plt.imread('/content/drive/MyDrive/SB - Icon Lockup - Colour positive.png')
# Calculate desired dimensions while maintaining aspect ratio
logo_aspect = 5885/943  # original aspect ratio
logo_width = 0.15  # desired width as percentage of figure
logo_height = logo_width/logo_aspect
# Create new axes for the logo
logo_ax = fig.add_axes([0.75, 0.02, logo_width, logo_height])
logo_ax.imshow(logo)
logo_ax.axis('off')  # hide axes

# Add title and subtitle
fig_text(
    x=0.05, y=.93,
    s="Goals vs Expected Goals",
    va="bottom", ha="left",
    fontsize=13, font=font_semi, color="darkred"
)
fig_text(
    x=0.05, y=0.90,
    s="Analysis of the difference between expected goals and actual goals scored by team",
    va="bottom", ha="left",
    fontsize=8, font=font_med, color="darkorange"
)

plt.show()

## Focus on players: shots and key passes



In [ ]:
import os

# Function to generate a semicircle with center (h, k) and radius r
def semicircle(r, h, k):
    x0 = h - r  # Starting horizontal point
    x1 = h + r  # Ending horizontal point
    x = np.linspace(x0, x1, 1000)  # Generate 1000 points on the x-axis
    y = k - np.sqrt(r**2 - (x - h)**2)  # Calculate y values using the circle equation
    return x, y

def plot_hexbin_shot(ax, player_data, player_stats, name_map, team_logos_dir):
    """
    Create a hexbin shot map for a player with associated statistics

    Parameters:
    - ax: Matplotlib axis to plot on
    - player_data: DataFrame with player shot data
    - player_stats: Series with player statistics
    - name_map: Dictionary to map full names to display names
    - team_logos_dir: Directory containing team logo images
    """
    pitch = VerticalPitch(pitch_type='statsbomb', goal_type='box', linewidth=2,
                          line_color='black', pitch_color='blanchedalmond', half=True, pad_top=20)
    pitch.draw(ax=ax)

    # Create hexbin plot of shots
    bins = pitch.hexbin(player_data['location.x'], player_data['location.y'], ax=ax,
                         cmap='SOC4', gridsize=(14,14), edgecolors='black', alpha=0.75, linewidth=0.25)

    # Calculate median distance from goal (converted to meters)
    median_dist = (120 - player_data['location.x'].median()) * 0.9144
    x_circle, y_circle = semicircle(median_dist, 40, 120)
    ax.plot(x_circle, y_circle, ls='--', color='darkred', lw=2, zorder=5)

    # Add arrow indicator
    ax.annotate('',
                xy=(50, 125),
                xytext=(40 - median_dist, 125),
                arrowprops=dict(arrowstyle='<-', color='darkred', lw=1.5))

    # Define positions and labels for statistics hexagons
    annot_x = [10, 25, 55, 70]
    annot_texts = ['Goals', 'xG', 'Shots', 'xG/Shots']
    annot_stats = [int(player_stats['goals']), player_stats['xG'], int(player_stats['shots']), player_stats.get('xG/shot', 0)]

    # Create hexagons for each statistic with its value
    for x, text, stat in zip(annot_x, annot_texts, annot_stats):
        hex_annotation = RegularPolygon((x, 70), numVertices=6, radius=4.5, edgecolor='black', facecolor='white', linewidth=1.5)
        ax.add_patch(hex_annotation)
        text_stat = f'{stat}' if isinstance(stat, int) else f'{stat:.2f}'
        text_ = ax.annotate(text_stat, xy=(x, 70), size=17, ha='center', va='center', color='black', fontproperties=font_med)
        text_.set_path_effects([path_effects.Stroke(linewidth=2, foreground='white'), path_effects.Normal()])
        ax.annotate(text, xy=(x, 63), size=17, ha='center', va='center', color='black', fontproperties=font_med)

    # Add player name
    original_name = player_data['player.name'].iloc[0]
    display_name = name_map.get(original_name, original_name)

    text_ = ax.annotate(display_name, xy=(40, 131), size=27, color='black', ha='center', va='center', zorder=100, fontproperties=font_semi)
    text_.set_path_effects([path_effects.Stroke(linewidth=3, foreground='white'), path_effects.Normal()])

    # Add median distance label
    text_ = ax.annotate(f"{median_dist:.1f}m", xy=(40 - median_dist - 2, 125), size=19, color='darkred', ha='right', va='center', weight='bold', zorder=100, fontproperties=font_med)
    text_.set_path_effects([path_effects.Stroke(linewidth=3, foreground='white'), path_effects.Normal()])

    ax.annotate("Median Distance", xy=(63, 125), size=20, color='darkred', ha='center', va='center', weight='bold', zorder=100, fontproperties=font_med)

    # Add team logo
    team_name = player_data['team.name'].iloc[0]
    logo_path = os.path.join(team_logos_dir, f"{team_name}.png")
    if os.path.exists(logo_path):
        logo = plt.imread(logo_path)
        ax_logo = ax.inset_axes([0.05, 0.82, 0.1, 0.1], anchor='NE', zorder=1)
        ax_logo.imshow(logo)
        ax_logo.axis('off')

    return ax

def plot_all_top_scorers(shots_df, name_map, team_logos_dir, n_players=6):
    """
    Create a visualization of shot maps for the top goal scorers

    Parameters:
    - shots_df: DataFrame containing all shots
    - name_map: Dictionary to map full names to display names
    - team_logos_dir: Directory containing team logo images
    - n_players: Number of top scorers to display (default 6)
    """
    # Calculate statistics for each player
    player_stats = shots_df.groupby('player.name').agg({'shot.outcome.name': lambda x: sum(x == 'Goal'),
                                                         'shot.statsbomb_xg': 'sum',
                                                         'player.name': 'count'})
    player_stats.rename(columns={'shot.outcome.name': 'goals', 'shot.statsbomb_xg': 'xG', 'player.name': 'shots'}, inplace=True)
    player_stats['xG/shot'] = (player_stats['xG'] / player_stats['shots']).round(2).fillna(0)

    # Get top n players by goals scored
    top_players = player_stats.nlargest(n_players, 'goals')

    # Create figure and grid for subplots
    fig = plt.figure(figsize=(20, 12), dpi=300, facecolor='blanchedalmond')
    gs = gridspec.GridSpec(2, 3, figure=fig)
    fig.suptitle("TOP SCORERS SERIE A 2015/16", fontsize=40, color='darkred', fontproperties=font_semi)

    # Create individual shot maps for each top player
    for idx, player_name in enumerate(top_players.index):
        row, col = divmod(idx, 3)
        ax = fig.add_subplot(gs[row, col])
        player_shots = shots_df[shots_df['player.name'] == player_name]
        player_stat = top_players.loc[player_name]
        plot_hexbin_shot(ax, player_shots, player_stat, name_map, team_logos_dir)

    # Add StatsBomb logo
    logo = plt.imread('/content/drive/MyDrive/SB - Icon Lockup - Colour positive.png')
    logo_ax = fig.add_axes([0.85, 0.91, 0.09, 0.1])
    logo_ax.imshow(logo)
    logo_ax.axis('off')

    plt.tight_layout()
    plt.savefig("topScorers.png", bbox_inches='tight', dpi=300)
    plt.show()

# Dictionary to map full player names to display names
name_map = {
    "Paulo Bruno Exequiel Dybala": "Paulo Dybala",
    "Gonzalo Gerardo Higuaín": "Gonzalo Higuaín",
    "Mauro Emanuel Icardi Rivero": "Mauro Icardi",
    "Carlos Arturo Bacca Ahumada": "Carlos Bacca"
}

# Directory containing team logos
team_logos_dir = "/content/drive/MyDrive/LoghiITA"

# Filter events to include only shots
shots_df = df_events[df_events['type.name'] == 'Shot'].copy()

# Create and display the visualization
plot_all_top_scorers(shots_df, name_map, team_logos_dir)

In [ ]:
def plot_key_pass_map(ax, player_data, total_kp, name_map, team_logos_dir):
    pitch = VerticalPitch(pitch_type='statsbomb', goal_type='box', linewidth=2,
                          line_color='black', pitch_color='blanchedalmond', half=True, pad_top=20)
    pitch.draw(ax=ax)

    assists = player_data[player_data['pass.goal_assist'] == True]
    key_passes = player_data[player_data['pass.goal_assist'] != True]

    pitch.lines(key_passes['location.x'], key_passes['location.y'],
                key_passes['pass.end_location.x'], key_passes['pass.end_location.y'],
                lw=2, comet=True, color='darkred', alpha=0.15, ax=ax)

    pitch.lines(assists['location.x'], assists['location.y'],
                assists['pass.end_location.x'], assists['pass.end_location.y'],
                lw=3, comet=True, color='darkred', alpha=1, ax=ax)

    original_name = player_data['player.name'].iloc[0]
    display_name = name_map.get(original_name, original_name)

    text_ = ax.annotate(display_name, xy=(40, 131), size=27, ha='center', va='center',
                        color='black', fontweight='bold', fontproperties=font_semi)
    text_.set_path_effects([path_effects.Stroke(linewidth=3, foreground='white'), path_effects.Normal()])

    total_assists = len(assists)

    ax.annotate("Key passes:", xy=(25, 126), size=20, ha='center', va='center',
                color='darkred', fontproperties=font_med)
    text_kp = ax.annotate(str(total_kp), xy=(36, 125.9), size=18, ha='center', va='center',
                          color='darkred', fontproperties=font_med)
    text_kp.set_path_effects([path_effects.Stroke(linewidth=3, foreground='white'), path_effects.Normal()])

    ax.annotate("Assists:", xy=(55, 126), size=20, ha='center', va='center',
                color='darkred', fontproperties=font_med)
    text_assist = ax.annotate(str(total_assists), xy=(62, 125.9), size=18, ha='center', va='center',
                               color='darkred', fontproperties=font_med)
    text_assist.set_path_effects([path_effects.Stroke(linewidth=3, foreground='white'), path_effects.Normal()])

    team_name = player_data['team.name'].iloc[0]
    logo_path = os.path.join(team_logos_dir, f"{team_name}.png")
    if os.path.exists(logo_path):
        logo = plt.imread(logo_path)
        ax_logo = ax.inset_axes([0.05, 0.82, 0.1, 0.1], anchor='NE', zorder=1)
        ax_logo.imshow(logo)
        ax_logo.axis('off')

    return ax

def plot_top_key_passers(events_df, name_map, team_logos_dir, n_players=6):
    key_passes_df = events_df[
        (events_df['type.name'] == 'Pass') &
        ((events_df['pass.shot_assist'] == True) |
         (events_df['pass.goal_assist'] == True) |
         (~events_df['pass.assisted_shot_id'].isna()))
    ]

    player_key_passes = key_passes_df.groupby('player.name').size().sort_values(ascending=False)
    top_players = player_key_passes.head(n_players)

    fig = plt.figure(figsize=(20, 12), dpi=300, facecolor='blanchedalmond')
    gs = gridspec.GridSpec(2, 3, figure=fig)
    fig.suptitle("TOP KEY PASSERS SERIE A 2015/16", fontsize=40, color='darkred', fontproperties=font_semi)

    for idx, (player, total_kp) in enumerate(top_players.items()):
        row, col = divmod(idx, 3)
        ax = fig.add_subplot(gs[row, col])
        player_data = key_passes_df[key_passes_df['player.name'] == player]
        plot_key_pass_map(ax, player_data, total_kp, name_map, team_logos_dir)

    logo = plt.imread('/content/drive/MyDrive/SB - Icon Lockup - Colour positive.png')
    logo_ax = fig.add_axes([0.85, 0.91, 0.09, 0.1])
    logo_ax.imshow(logo)
    logo_ax.axis('off')

    plt.tight_layout()
    plt.savefig("topPassers.png", bbox_inches='tight', dpi=300)
    plt.show()

name_map = {
    "Paulo Bruno Exequiel Dybala": "Paulo Dybala",
    "Marek Hamsik": "Marek Hamsik",
    "Borja Valero Iglesias": "Borja Valero",
    "Miralem Pjanic": "Miralem Pjanic",
    "Riccardo Saponara" : "Riccardo Saponara",
    "Alejandro Darío Gómez" : "Alejandro Gómez"
}

team_logos_dir = "/content/drive/MyDrive/LoghiITA"

plot_top_key_passers(df_events, name_map, team_logos_dir)


## Analysis of passing accuracy by field zone



In [ ]:
team_to_logo = {
   'AC Milan': 'AC Milan.png',
   'Atalanta': 'Atalanta.png',
   'Bologna': 'Bologna.png',
   'Carpi': 'Carpi.png',
   'Chievo': 'Chievo.png',
   'Empoli': 'Empoli.png',
   'Fiorentina': 'Fiorentina.png',
   'Frosinone': 'Frosinone.png',
   'Genoa': 'Genoa.png',
   'Inter Milan': 'Inter Milan.png',
   'Juventus': 'Juventus.png',
   'Lazio': 'Lazio.png',
   'Napoli': 'Napoli.png',
   'Palermo': 'Palermo.png',
   'AS Roma': 'AS Roma.png',
   'Sampdoria': 'Sampdoria.png',
   'Sassuolo': 'Sassuolo.png',
   'Torino': 'Torino.png',
   'Udinese': 'Udinese.png',
   'Hellas Verona': 'Verona.png'
}

def plot_all_teams_passing_zones(classifica):
    """
    Creates visualizations of passing accuracy by field zone for all teams.

    Parameters:
    classifica (DataFrame): DataFrame containing team statistics including passing accuracy by zone
    """
    # Collect all passing accuracy percentages for global normalization
    all_percentages = []
    for team_name in classifica.index:
        team_data = classifica.loc[team_name]
        all_percentages.extend([
            team_data['defensive_passes_pct'],
            team_data['middle_passes_pct'],
            team_data['attacking_passes_pct']
        ])

    # Create a global normalization for color mapping
    norm = plt.Normalize(vmin=min(all_percentages), vmax=max(all_percentages))

    # Create figure
    fig = plt.figure(figsize=(20,16), dpi=300, facecolor='blanchedalmond')

    # Add title area
    title_ax = fig.add_axes([0, 0.95, 1, 0.05])
    title_ax.axis('off')
    title_ax.set_facecolor('blanchedalmond')

    # Add title and subtitle
    title_ax.text(0.0, 1.7, "Passing Accuracy by Field Zone",
                 fontsize=35, color="darkred", fontproperties=font_semi)
    title_ax.text(0.0, 1.3, "Analysis of passing accuracy in the three zones of the field for each team",
                 fontsize=25, color="darkorange", fontproperties=font_med)

    # For each team in the dataset
    for idx, team_name in enumerate(classifica.index):
        # Create a subplot in the 5x4 grid
        ax = plt.subplot(4, 5, idx+1)
        # Create the pitch
        pitch = Pitch(
            pitch_type='statsbomb',
            goal_type='box',
            linewidth=1.5,
            line_color='black',
            pitch_color='blanchedalmond',
            half=False
        )
        pitch.draw(ax=ax)

        # Get team data
        team_data = classifica.loc[team_name]
        percentages = [
            team_data['defensive_passes_pct'],
            team_data['middle_passes_pct'],
            team_data['attacking_passes_pct']
        ]

        # Define pitch divisions (defensive, middle, attacking thirds)
        pitch_divisions = [0, 40, 80, 120]

        # Color each zone based on passing accuracy
        for index, x in enumerate(pitch_divisions[:-1]):
            mid_point = (pitch_divisions[index] + pitch_divisions[index + 1])/2

            # Fill zone with color based on accuracy
            ax.fill_between(
                x=[pitch_divisions[index], pitch_divisions[index + 1]],
                y1=0, y2=80,
                color=soc_cm(norm(percentages[index])),
                zorder=-1, alpha=0.75
            )

            # Add white box for percentage
            rect = plt.Rectangle(
                (mid_point-10, 3),  # position of the box
                20, 10,              # width and height of the box
                facecolor='white',
                edgecolor='black',
                linewidth=1.6,
                zorder=2
            )
            ax.add_patch(rect)

            # Add percentage in the box
            ax.annotate(
                xy=(mid_point, 8),  # text position (center of box)
                text=f"{percentages[index]:.1f}%",
                annotation_clip=False,
                ha='center',
                va='center',
                weight='bold',
                size=15,
                fontproperties=font_med,
                color="darkorange",
                zorder=3
            )

        # Team name
        text_ = ax.annotate(
              xy=(60, -18),
              text=f"{team_name.upper()}",
              size=22,
              color='darkred',
              ha='center',
              va='center',
              annotation_clip=False,
              fontproperties=font_semi
          )

        # Add white outline to text
        text_.set_path_effects([
              path_effects.Stroke(linewidth=3, foreground='white'),  # Thick white outline
              path_effects.Normal()
          ])

        # Total accuracy percentage
        total_success = team_data['total_passes_pct']
        ax.annotate(
            xy=(60,-8),
            text=f"Total passing accuracy: {total_success:.1f}%",
            size=16,
            color='black',
            ha='center',
            va='center',
            annotation_clip=False,
            fontproperties=font_med
        )

        # Add team logo
        if team_name in team_to_logo:
            logo_path = f'/content/drive/MyDrive/LoghiITA/{team_to_logo[team_name]}'
            try:
                logo = plt.imread(logo_path)
                # Calculate relative positions for each subplot
                row = idx // 5  # row in grid (0-3)
                col = idx % 5   # column in grid (0-4)

                # Modify position and dimensions
                logo_ax = fig.add_axes([
                    0.16 + (col * 0.197),    # x position
                    0.952 - (row * 0.24),     # y position
                    0.05,                    # width
                    0.025                    # height
                ])
                logo_ax.imshow(logo)
                logo_ax.axis('off')
            except:
                print(f"Logo not found for {team_name}")

    # Add StatsBomb logo
    logo = plt.imread('/content/drive/MyDrive/SB - Icon Lockup - Colour positive.png')
    logo_ax = fig.add_axes([0.85, 0.02, 0.1, 0.02])
    logo_ax.imshow(logo)
    logo_ax.axis('off')

    plt.tight_layout()
    plt.show()

# Plot all teams
plot_all_teams_passing_zones(classifica)

## Pressing intensity and effectiveness


In [ ]:
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

def getImage(path):
    return OffsetImage(plt.imread(path), zoom=0.3, alpha=1)

fig, ax = plt.subplots(figsize=(10, 10), facecolor='blanchedalmond')  # Figure size and background color
plt.gca().set_facecolor('blanchedalmond')

# Create an invisible scatter plot to define the axis limits
ax.scatter(classifica["ppda"], classifica["opp_pass_accuracy"], color='blanchedalmond', s=100, alpha=0)

# Add team logos
for index, row in classifica.iterrows():
    team_name = row.name  # Assuming the index of the dataframe is the team name
    logo_path = team_to_logo.get(team_name, None)  # Get the logo path
    if logo_path:
        ab = AnnotationBbox(getImage(f'/content/drive/MyDrive/LoghiITA/{logo_path}'),
                            (row["ppda"], row["opp_pass_accuracy"]), frameon=False)
        ax.add_artist(ab)

# Invert both axes
ax.invert_xaxis()
ax.invert_yaxis()

# Title and subtitle
plt.title('Pressing intensity and effectiveness', x = -0.05, fontproperties=font_semi, fontsize=26, color="darkred", pad=50, loc='left')
fig.text(0.085, 0.915, "Analysis of the relationship between PPDA and opponent pass accuracy",
         fontproperties=font_semi, fontsize=18, color="darkorange")

# Axis labels and ticks
plt.xticks(fontproperties=font_normal, fontsize=16, color='black')
plt.yticks(fontproperties=font_normal, fontsize=16, color='black')
ax.set_xlabel('PPDA', fontproperties=font_med, fontsize=18, color="black")
ax.set_ylabel('Opponent Pass Accuracy (%)', fontproperties=font_med, fontsize=18, color="black")

# Axis styling
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.grid(True, color="white", linestyle='--', linewidth=0.5)

# Add average lines
ax.axvline(x=classifica["ppda"].mean(), color='black', linestyle='--', label='Average PPDA', alpha=0.5)
ax.axhline(y=classifica["opp_pass_accuracy"].mean(), color='black', linestyle='--', label='Average Accuracy', alpha=0.5)

# Descriptive quadrant labels
fig.text(0.40, 0.8, "Not intense\nbut effective pressing", fontsize=14, color="grey", alpha=0.6, fontproperties=font_med)
fig.text(0.6, 0.8, "Intense\nand effective pressing", fontsize=14, color="grey", alpha=0.6, fontproperties=font_med)
fig.text(0.40, 0.2, "Not intense\nand ineffective pressing", fontsize=14, color="grey", alpha=0.6, fontproperties=font_med)
fig.text(0.6, 0.2, "Intense\nbut ineffective pressing", fontsize=14, color="grey", alpha=0.6, fontproperties=font_med)

# Add logo to the bottom right corner
logo = plt.imread('/content/drive/MyDrive/SB - Icon Lockup - Colour positive.png')
logo_ax = fig.add_axes([0.80, 0.02, 0.1, 0.02])
logo_ax.imshow(logo)
logo_ax.axis('off')

# Display the plot
plt.show()


In [ ]:
def plot_all_teams_pressure_zones(events_df, classifica):
    # Filter only pressure events
    pressure_events = events_df[events_df['type.name'] == 'Pressure']

    # Create the figure
    fig = plt.figure(figsize=(20,16), dpi=300, facecolor='blanchedalmond')

    # Axis for titles
    title_ax = fig.add_axes([0, 0.95, 1, 0.05])
    title_ax.axis('off')
    title_ax.set_facecolor('blanchedalmond')

    # Add titles
    title_ax.text(0.0, 1.7, "Team Pressure Zones",
                 fontsize=35, color="darkred", fontproperties=font_semi)
    title_ax.text(0.0, 1.3, "Analysis of the spatial distribution of pressure for each team",
                 fontsize=25, color="darkorange", fontproperties=font_med)

    # For each team
    for idx, team_name in enumerate(classifica.index):
        # Subplot in a 4x5 grid
        ax = plt.subplot(4, 5, idx+1)

        # Create the pitch
        pitch = Pitch(
            pitch_type='statsbomb',
            goal_type='box',
            linewidth=1.5,
            line_color='black',
            pitch_color='blanchedalmond',
            half=False
        )
        pitch.draw(ax=ax)

        # Filter pressure events of the current team
        team_pressure = pressure_events[pressure_events['team.name'] == team_name]

        # Create the heatmap
        if not team_pressure.empty:
            pitch.kdeplot(
                team_pressure['location.x'],
                team_pressure['location.y'],
                ax=ax,
                cmap=soc_cm,
                alpha=0.5,
                levels=50, zorder=-1
            )

        # Line for the median height of pressure
        median_x = classifica.loc[team_name, 'pressure_height']
        ax.axvline(x=median_x, color='grey', linestyle='--', linewidth=2.5, alpha=1)

        # Team name
        text_ = ax.annotate(
            xy=(60, -18),
            text=f"{team_name.upper()}",
            size=22,
            color='darkred',
            ha='center',
            va='center',
            annotation_clip=False,
            fontproperties=font_semi
        )

        text_.set_path_effects([
            path_effects.Stroke(linewidth=3, foreground='white'),
            path_effects.Normal()
        ])

        # Median pressure height in meters
        median_height_meters = classifica.loc[team_name, 'pressure_height'] * 0.9144
        ax.annotate(
            xy=(60,-8),
            text=f"Median Height: {median_height_meters:.1f}m",
            size=16,
            color='black',
            ha='center',
            va='center',
            annotation_clip=False,
            fontproperties=font_med
        )

        # Team logos
        if team_name in team_to_logo:
            logo_path = f'/content/drive/MyDrive/LoghiITA/{team_to_logo[team_name]}'
            try:
                logo = plt.imread(logo_path)
                row = idx // 5
                col = idx % 5
                logo_ax = fig.add_axes([
                    0.16 + (col * 0.197),
                    0.952 - (row * 0.24),
                    0.05,
                    0.025
                ])
                logo_ax.imshow(logo)
                logo_ax.axis('off')
            except:
                print(f"Logo not found for {team_name}")

    # StatsBomb logo
    logo = plt.imread('/content/drive/MyDrive/SB - Icon Lockup - Colour positive.png')
    logo_ax = fig.add_axes([0.85, 0.02, 0.1, 0.02])
    logo_ax.imshow(logo)
    logo_ax.axis('off')

    plt.tight_layout()
    plt.savefig("pressurezone.png", bbox_inches='tight', dpi=300)
    plt.show()

plot_all_teams_pressure_zones(df_events, classifica)


# Creation of the Match_Results Table

To analyze how the first half influences the final result of a match, we're creating a new table (`match_results`) that will contain detailed information about each match. This table will represent the foundation for our comparative study between the first half and the complete match.

The table will contain for each match:
- Match identifier and teams involved
- Goals and result of the first half
- Goals and final result
- Statistics of the first half
- Statistics of the complete match

Through this data structuring, we will be able to:
- Analyze cases of result reversals
- Compare the progression of statistics between first and second half
- Identify patterns that may suggest how predictive the first half is of the final result

This data preparation phase is fundamental for the subsequent statistical analyses and for the creation of predictive models.

In [ ]:
def create_base_match_results(events_df):
    # Create a list of dictionaries, one for each match
    match_results = []

    for match_id in events_df['match_id'].unique():
        # Filter events for the current match
        match_events = events_df[events_df['match_id'] == match_id]

        # Get the two teams
        teams = match_events['team.name'].unique()
        home_team, away_team = teams[0], teams[1]

        # Compute first-half goals, including all types of goals
        ht_goals = match_events[
            (
                # Shots resulting in a goal
                ((match_events['type.name'] == 'Shot') &
                 (match_events['shot.outcome.name'] == 'Goal')) |
                # Own goals
                (match_events['type.name'].isin(['Own Goal For']))
            ) &
            (match_events['period'] == 1)
        ].groupby('team.name').size()

        ht_home_goals = ht_goals.get(home_team, 0)
        ht_away_goals = ht_goals.get(away_team, 0)

        # Compute total goals, including all types of goals
        ft_goals = match_events[
            # Shots resulting in a goal
            ((match_events['type.name'] == 'Shot') &
             (match_events['shot.outcome.name'] == 'Goal')) |
            # Own goals
            (match_events['type.name'].isin(['Own Goal For']))
        ].groupby('team.name').size()

        ft_home_goals = ft_goals.get(home_team, 0)
        ft_away_goals = ft_goals.get(away_team, 0)

        # Determine results (1/X/2)
        ht_result = '1' if ht_home_goals > ht_away_goals else '2' if ht_home_goals < ht_away_goals else 'X'
        ft_result = '1' if ft_home_goals > ft_away_goals else '2' if ft_home_goals < ft_away_goals else 'X'

        # Append the row to the list
        match_results.append({
            'match_id': match_id,
            'home_team': home_team,
            'away_team': away_team,
            'ht_home_goals': ht_home_goals,
            'ht_away_goals': ht_away_goals,
            'ft_home_goals': ft_home_goals,
            'ft_away_goals': ft_away_goals,
            'ht_result': ht_result,
            'ft_result': ft_result
        })

    # Convert the list into a DataFrame
    match_results_df = pd.DataFrame(match_results)
    return match_results_df

match_results = create_base_match_results(df_events)

In [ ]:
def add_shooting_stats(events_df, match_results_df):
    for match_id in match_results_df['match_id']:
        match_events = events_df[events_df['match_id'] == match_id]
        home_team = match_results_df.loc[match_results_df['match_id'] == match_id, 'home_team'].iloc[0]
        away_team = match_results_df.loc[match_results_df['match_id'] == match_id, 'away_team'].iloc[0]

        # First-half shooting stats
        ht_shots = match_events[
            (match_events['type.name'] == 'Shot') &
            (match_events['period'] == 1)
        ]

        # Total first-half shots
        ht_home_shots = len(ht_shots[ht_shots['team.name'] == home_team])
        ht_away_shots = len(ht_shots[ht_shots['team.name'] == away_team])

        # First-half shots on target
        ht_home_shots_on_target = len(ht_shots[
            (ht_shots['team.name'] == home_team) &
            (ht_shots['shot.outcome.name'].isin(['Goal', 'Saved', 'Saved to Post']))
        ])
        ht_away_shots_on_target = len(ht_shots[
            (ht_shots['team.name'] == away_team) &
            (ht_shots['shot.outcome.name'].isin(['Goal', 'Saved', 'Saved to Post']))
        ])

        # First-half xG
        ht_home_xg = ht_shots[ht_shots['team.name'] == home_team]['shot.statsbomb_xg'].sum()
        ht_away_xg = ht_shots[ht_shots['team.name'] == away_team]['shot.statsbomb_xg'].sum()

        # Same stats for the full match
        ft_shots = match_events[match_events['type.name'] == 'Shot']

        ft_home_shots = len(ft_shots[ft_shots['team.name'] == home_team])
        ft_away_shots = len(ft_shots[ft_shots['team.name'] == away_team])

        ft_home_shots_on_target = len(ft_shots[
            (ft_shots['team.name'] == home_team) &
            (ft_shots['shot.outcome.name'].isin(['Goal', 'Saved', 'Saved to Post']))
        ])
        ft_away_shots_on_target = len(ft_shots[
            (ft_shots['team.name'] == away_team) &
            (ft_shots['shot.outcome.name'].isin(['Goal', 'Saved', 'Saved to Post']))
        ])

        ft_home_xg = ft_shots[ft_shots['team.name'] == home_team]['shot.statsbomb_xg'].sum()
        ft_away_xg = ft_shots[ft_shots['team.name'] == away_team]['shot.statsbomb_xg'].sum()

        # Update the DataFrame
        match_results_df.loc[match_results_df['match_id'] == match_id, [
            'ht_home_shots', 'ht_away_shots',
            'ht_home_shots_on_target', 'ht_away_shots_on_target',
            'ht_home_xg', 'ht_away_xg',
            'ft_home_shots', 'ft_away_shots',
            'ft_home_shots_on_target', 'ft_away_shots_on_target',
            'ft_home_xg', 'ft_away_xg'
        ]] = [
            ht_home_shots, ht_away_shots,
            ht_home_shots_on_target, ht_away_shots_on_target,
            ht_home_xg, ht_away_xg,
            ft_home_shots, ft_away_shots,
            ft_home_shots_on_target, ft_away_shots_on_target,
            ft_home_xg, ft_away_xg
        ]

    return match_results_df

match_results = add_shooting_stats(df_events, match_results)

In [ ]:
def add_possession_stats(events_df, match_results_df):
    for match_id in match_results_df['match_id']:
        match_events = events_df[events_df['match_id'] == match_id]
        home_team = match_results_df.loc[match_results_df['match_id'] == match_id, 'home_team'].iloc[0]
        away_team = match_results_df.loc[match_results_df['match_id'] == match_id, 'away_team'].iloc[0]

        # First-half statistics
        ht_events = match_events[match_events['period'] == 1]

        # Ball possession (based on passes)
        ht_passes = ht_events[ht_events['type.name'] == 'Pass']
        ht_home_passes = len(ht_passes[ht_passes['team.name'] == home_team])
        ht_away_passes = len(ht_passes[ht_passes['team.name'] == away_team])

        # Possession percentage calculation
        ht_total_passes = ht_home_passes + ht_away_passes
        ht_home_possession = (ht_home_passes / ht_total_passes * 100) if ht_total_passes > 0 else 0
        ht_away_possession = (ht_away_passes / ht_total_passes * 100) if ht_total_passes > 0 else 0

        # Completed passes
        ht_home_completed = len(ht_passes[
            (ht_passes['team.name'] == home_team) &
            (ht_passes['pass.outcome.name'].isna())  # completed passes have no outcome
        ])
        ht_away_completed = len(ht_passes[
            (ht_passes['team.name'] == away_team) &
            (ht_passes['pass.outcome.name'].isna())
        ])

        # Field Tilt (passes in the final third)
        ht_home_final_third = len(ht_passes[
            (ht_passes['team.name'] == home_team) &
            (ht_passes['location.x'] > 80)
        ])
        ht_away_final_third = len(ht_passes[
            (ht_passes['team.name'] == away_team) &
            (ht_passes['location.x'] > 80)
        ])

        ht_total_final_third = ht_home_final_third + ht_away_final_third
        ht_home_field_tilt = (ht_home_final_third / ht_total_final_third * 100) if ht_total_final_third > 0 else 0
        ht_away_field_tilt = (ht_away_final_third / ht_total_final_third * 100) if ht_total_final_third > 0 else 0

        # Same stats for the full match
        ft_passes = match_events[match_events['type.name'] == 'Pass']
        ft_home_passes = len(ft_passes[ft_passes['team.name'] == home_team])
        ft_away_passes = len(ft_passes[ft_passes['team.name'] == away_team])

        ft_total_passes = ft_home_passes + ft_away_passes
        ft_home_possession = (ft_home_passes / ft_total_passes * 100) if ft_total_passes > 0 else 0
        ft_away_possession = (ft_away_passes / ft_total_passes * 100) if ft_total_passes > 0 else 0

        ft_home_completed = len(ft_passes[
            (ft_passes['team.name'] == home_team) &
            (ft_passes['pass.outcome.name'].isna())
        ])
        ft_away_completed = len(ft_passes[
            (ft_passes['team.name'] == away_team) &
            (ft_passes['pass.outcome.name'].isna())
        ])

        ft_home_final_third = len(ft_passes[
            (ft_passes['team.name'] == home_team) &
            (ft_passes['location.x'] > 80)
        ])
        ft_away_final_third = len(ft_passes[
            (ft_passes['team.name'] == away_team) &
            (ft_passes['location.x'] > 80)
        ])

        ft_total_final_third = ft_home_final_third + ft_away_final_third
        ft_home_field_tilt = (ft_home_final_third / ft_total_final_third * 100) if ft_total_final_third > 0 else 0
        ft_away_field_tilt = (ft_away_final_third / ft_total_final_third * 100) if ft_total_final_third > 0 else 0

        # Update the DataFrame
        match_results_df.loc[match_results_df['match_id'] == match_id, [
            'ht_home_possession', 'ht_away_possession',
            'ht_home_passes', 'ht_away_passes',
            'ht_home_passes_completed', 'ht_away_passes_completed',
            'ht_home_field_tilt', 'ht_away_field_tilt',
            'ft_home_possession', 'ft_away_possession',
            'ft_home_passes', 'ft_away_passes',
            'ft_home_passes_completed', 'ft_away_passes_completed',
            'ft_home_field_tilt', 'ft_away_field_tilt'
        ]] = [
            ht_home_possession, ht_away_possession,
            ht_home_passes, ht_away_passes,
            ht_home_completed, ht_away_completed,
            ht_home_field_tilt, ht_away_field_tilt,
            ft_home_possession, ft_away_possession,
            ft_home_passes, ft_away_passes,
            ft_home_completed, ft_away_completed,
            ft_home_field_tilt, ft_away_field_tilt
        ]

    return match_results_df

# Update the DataFrame
add_possession_stats(df_events, match_results)

In [ ]:
def add_defensive_stats(events_df, match_results_df):
   for match_id in match_results_df['match_id']:
       match_events = events_df[events_df['match_id'] == match_id]
       home_team = match_results_df.loc[match_results_df['match_id'] == match_id, 'home_team'].iloc[0]
       away_team = match_results_df.loc[match_results_df['match_id'] == match_id, 'away_team'].iloc[0]

       # First-half statistics
       ht_events = match_events[match_events['period'] == 1]

       # Tackles + Duels 50/50)
       ht_home_duels_won = len(ht_events[
           (ht_events['team.name'] == home_team) &
           (
               (
                   (ht_events['type.name'] == 'Duel') &
                   (ht_events['duel.type.name'] == 'Tackle') &
                   (ht_events['duel.outcome.name'].isin(['Success In Play', 'Won', 'Success Out']))
               ) |
               (
                   (ht_events['type.name'] == '50/50') &
                   (ht_events['50_50.outcome.name'].isin(['Won', 'Success To Team']))
               )
           )
       ])

       ht_away_duels_won = len(ht_events[
           (ht_events['team.name'] == away_team) &
           (
               (
                   (ht_events['type.name'] == 'Duel') &
                   (ht_events['duel.type.name'] == 'Tackle') &
                   (ht_events['duel.outcome.name'].isin(['Success In Play', 'Won', 'Success Out']))
               ) |
               (
                   (ht_events['type.name'] == '50/50') &
                   (ht_events['50_50.outcome.name'].isin(['Won', 'Success To Team']))
               )
           )
       ])

       # Int & Blk
       ht_home_interceptions_blocks = len(ht_events[
           (ht_events['team.name'] == home_team) &
           (
               (
                   (ht_events['type.name'] == 'Interception') &
                   (ht_events['interception.outcome.name'].isin(['Won', 'Success In Play', 'Success Out']))
               ) |
               (
                   (ht_events['type.name'] == 'Block') &
                   (ht_events['block.save_block'].isna())
               )
           )
       ])

       ht_away_interceptions_blocks = len(ht_events[
           (ht_events['team.name'] == away_team) &
           (
               (
                   (ht_events['type.name'] == 'Interception') &
                   (ht_events['interception.outcome.name'].isin(['Won', 'Success In Play', 'Success Out']))
               ) |
               (
                   (ht_events['type.name'] == 'Block') &
                   (ht_events['block.save_block'].isna())
               )
           )
       ])

       # Clearance
       ht_home_clearances = len(ht_events[
           (ht_events['team.name'] == home_team) &
           (ht_events['type.name'] == 'Clearance')
       ])

       ht_away_clearances = len(ht_events[
           (ht_events['team.name'] == away_team) &
           (ht_events['type.name'] == 'Clearance')
       ])

       # Full time statistics
       # Tackle + Duels 50/50
       ft_home_duels_won = len(match_events[
           (match_events['team.name'] == home_team) &
           (
               (
                   (match_events['type.name'] == 'Duel') &
                   (match_events['duel.type.name'] == 'Tackle') &
                   (match_events['duel.outcome.name'].isin(['Success In Play', 'Won', 'Success Out']))
               ) |
               (
                   (match_events['type.name'] == '50/50') &
                   (match_events['50_50.outcome.name'].isin(['Won', 'Success To Team']))
               )
           )
       ])

       ft_away_duels_won = len(match_events[
           (match_events['team.name'] == away_team) &
           (
               (
                   (match_events['type.name'] == 'Duel') &
                   (match_events['duel.type.name'] == 'Tackle') &
                   (match_events['duel.outcome.name'].isin(['Success In Play', 'Won', 'Success Out']))
               ) |
               (
                   (match_events['type.name'] == '50/50') &
                   (match_events['50_50.outcome.name'].isin(['Won', 'Success To Team']))
               )
           )
       ])

       # Int & Blk
       ft_home_interceptions_blocks = len(match_events[
           (match_events['team.name'] == home_team) &
           (
               (
                   (match_events['type.name'] == 'Interception') &
                   (match_events['interception.outcome.name'].isin(['Won', 'Success In Play', 'Success Out']))
               ) |
               (
                   (match_events['type.name'] == 'Block') &
                   (match_events['block.save_block'].isna())
               )
           )
       ])

       ft_away_interceptions_blocks = len(match_events[
           (match_events['team.name'] == away_team) &
           (
               (
                   (match_events['type.name'] == 'Interception') &
                   (match_events['interception.outcome.name'].isin(['Won', 'Success In Play', 'Success Out']))
               ) |
               (
                   (match_events['type.name'] == 'Block') &
                   (match_events['block.save_block'].isna())
               )
           )
       ])

       # Clearance
       ft_home_clearances = len(match_events[
           (match_events['team.name'] == home_team) &
           (match_events['type.name'] == 'Clearance')
       ])

       ft_away_clearances = len(match_events[
           (match_events['team.name'] == away_team) &
           (match_events['type.name'] == 'Clearance')
       ])

       # Uptade DataFrame
       match_results_df.loc[match_results_df['match_id'] == match_id, [
           'ht_home_duels_won', 'ht_away_duels_won',
           'ht_home_interceptions_blocks', 'ht_away_interceptions_blocks',
           'ht_home_clearances', 'ht_away_clearances',
           'ft_home_duels_won', 'ft_away_duels_won',
           'ft_home_interceptions_blocks', 'ft_away_interceptions_blocks',
           'ft_home_clearances', 'ft_away_clearances'
       ]] = [
           ht_home_duels_won, ht_away_duels_won,
           ht_home_interceptions_blocks, ht_away_interceptions_blocks,
           ht_home_clearances, ht_away_clearances,
           ft_home_duels_won, ft_away_duels_won,
           ft_home_interceptions_blocks, ft_away_interceptions_blocks,
           ft_home_clearances, ft_away_clearances
       ]

   return match_results_df

add_defensive_stats(df_events, match_results)

In [ ]:
def add_ppda_stats(events_df, match_results_df):
    for match_id in match_results_df['match_id']:
        # Get all events for the current match
        match_events = events_df[events_df['match_id'] == match_id]

        # Identify home and away teams
        home_team = match_results_df.loc[match_results_df['match_id'] == match_id, 'home_team'].iloc[0]
        away_team = match_results_df.loc[match_results_df['match_id'] == match_id, 'away_team'].iloc[0]
        midfield = 60  # Midfield threshold for location on pitch

        # Filter first half events
        ht_events = match_events[match_events['period'] == 1]

        # --- Home PPDA (First Half) ---
        # Successful passes by the away team in their own half
        ht_away_passes = len(ht_events[
            (ht_events['type.name'] == 'Pass') &
            (ht_events['team.name'] == away_team) &
            (ht_events['location.x'] < midfield) &
            (ht_events['pass.outcome.name'].isna())
        ])

        # Defensive actions by home team in the offensive half
        ht_home_def_actions = len(ht_events[
            (ht_events['team.name'] == home_team) &
            (ht_events['location.x'] > midfield) &
            (
                # Successful tackles
                (
                    (ht_events['type.name'] == 'Duel') &
                    (ht_events['duel.type.name'] == 'Tackle') &
                    (ht_events['duel.outcome.name'].isin(['Success In Play', 'Won', 'Success Out']))
                ) |
                # Successful interceptions
                (
                    (ht_events['type.name'] == 'Interception') &
                    (ht_events['interception.outcome.name'].isin(['Won', 'Success In Play', 'Success Out']))
                ) |
                # Blocks (excluding goalkeeper save blocks)
                (
                    (ht_events['type.name'] == 'Block') &
                    (ht_events['block.save_block'].isna())
                ) |
                # All clearances
                (ht_events['type.name'] == 'Clearance') |
                # Successful 50/50 duels
                (
                    (ht_events['type.name'] == '50/50') &
                    (ht_events['50_50.outcome.name'].isin(['Won', 'Success To Team']))
                ) |
                # Fouls committed in offensive half
                (ht_events['type.name'] == 'Foul Committed')
            )
        ])

        # Calculate home PPDA for first half
        ht_home_ppda = round(ht_away_passes / ht_home_def_actions, 2) if ht_home_def_actions > 0 else None

        # --- Away PPDA (First Half) ---
        ht_home_passes = len(ht_events[
            (ht_events['type.name'] == 'Pass') &
            (ht_events['team.name'] == home_team) &
            (ht_events['location.x'] < midfield) &
            (ht_events['pass.outcome.name'].isna())
        ])

        ht_away_def_actions = len(ht_events[
            (ht_events['team.name'] == away_team) &
            (ht_events['location.x'] > midfield) &
            (
                (
                    (ht_events['type.name'] == 'Duel') &
                    (ht_events['duel.type.name'] == 'Tackle') &
                    (ht_events['duel.outcome.name'].isin(['Success In Play', 'Won', 'Success Out']))
                ) |
                (
                    (ht_events['type.name'] == 'Interception') &
                    (ht_events['interception.outcome.name'].isin(['Won', 'Success In Play', 'Success Out']))
                ) |
                (
                    (ht_events['type.name'] == 'Block') &
                    (ht_events['block.save_block'].isna())
                ) |
                (ht_events['type.name'] == 'Clearance') |
                (
                    (ht_events['type.name'] == '50/50') &
                    (ht_events['50_50.outcome.name'].isin(['Won', 'Success To Team']))
                ) |
                (ht_events['type.name'] == 'Foul Committed')
            )
        ])

        ht_away_ppda = round(ht_home_passes / ht_away_def_actions, 2) if ht_away_def_actions > 0 else None

        # --- Home PPDA (Full Match) ---
        ft_away_passes = len(match_events[
            (match_events['type.name'] == 'Pass') &
            (match_events['team.name'] == away_team) &
            (match_events['location.x'] < midfield) &
            (match_events['pass.outcome.name'].isna())
        ])

        ft_home_def_actions = len(match_events[
            (match_events['team.name'] == home_team) &
            (match_events['location.x'] > midfield) &
            (
                (
                    (match_events['type.name'] == 'Duel') &
                    (match_events['duel.type.name'] == 'Tackle') &
                    (match_events['duel.outcome.name'].isin(['Success In Play', 'Won', 'Success Out']))
                ) |
                (
                    (match_events['type.name'] == 'Interception') &
                    (match_events['interception.outcome.name'].isin(['Won', 'Success In Play', 'Success Out']))
                ) |
                (
                    (match_events['type.name'] == 'Block') &
                    (match_events['block.save_block'].isna())
                ) |
                (match_events['type.name'] == 'Clearance') |
                (
                    (match_events['type.name'] == '50/50') &
                    (match_events['50_50.outcome.name'].isin(['Won', 'Success To Team']))
                ) |
                (match_events['type.name'] == 'Foul Committed')
            )
        ])

        ft_home_ppda = round(ft_away_passes / ft_home_def_actions, 2) if ft_home_def_actions > 0 else None

        # --- Away PPDA (Full Match) ---
        ft_home_passes = len(match_events[
            (match_events['type.name'] == 'Pass') &
            (match_events['team.name'] == home_team) &
            (match_events['location.x'] < midfield) &
            (match_events['pass.outcome.name'].isna())
        ])

        ft_away_def_actions = len(match_events[
            (match_events['team.name'] == away_team) &
            (match_events['location.x'] > midfield) &
            (
                (
                    (match_events['type.name'] == 'Duel') &
                    (match_events['duel.type.name'] == 'Tackle') &
                    (match_events['duel.outcome.name'].isin(['Success In Play', 'Won', 'Success Out']))
                ) |
                (
                    (match_events['type.name'] == 'Interception') &
                    (match_events['interception.outcome.name'].isin(['Won', 'Success In Play', 'Success Out']))
                ) |
                (
                    (match_events['type.name'] == 'Block') &
                    (match_events['block.save_block'].isna())
                ) |
                (match_events['type.name'] == 'Clearance') |
                (
                    (match_events['type.name'] == '50/50') &
                    (match_events['50_50.outcome.name'].isin(['Won', 'Success To Team']))
                ) |
                (match_events['type.name'] == 'Foul Committed')
            )
        ])

        ft_away_ppda = round(ft_home_passes / ft_away_def_actions, 2) if ft_away_def_actions > 0 else None

        # Update match_results_df with all four PPDA values
        match_results_df.loc[match_results_df['match_id'] == match_id, [
            'ht_home_ppda', 'ht_away_ppda',
            'ft_home_ppda', 'ft_away_ppda'
        ]] = [
            ht_home_ppda, ht_away_ppda,
            ft_home_ppda, ft_away_ppda
        ]

    return match_results_df

add_ppda_stats(df_events, match_results)


In [ ]:
def add_pressure_height_stats(events_df, match_results_df):
    for match_id in match_results_df['match_id']:
        match_events = events_df[events_df['match_id'] == match_id]
        home_team = match_results_df.loc[match_results_df['match_id'] == match_id, 'home_team'].iloc[0]
        away_team = match_results_df.loc[match_results_df['match_id'] == match_id, 'away_team'].iloc[0]

        # Fist half statistics
        ht_events = match_events[match_events['period'] == 1]

        ht_pressure = ht_events[ht_events['type.name'] == 'Pressure']
        ht_home_pressure = ht_pressure[ht_pressure['team.name'] == home_team]['location.x'].median()
        ht_away_pressure = ht_pressure[ht_pressure['team.name'] == away_team]['location.x'].median()

        # Full time statistics
        ft_pressure = match_events[match_events['type.name'] == 'Pressure']
        ft_home_pressure = ft_pressure[ft_pressure['team.name'] == home_team]['location.x'].median()
        ft_away_pressure = ft_pressure[ft_pressure['team.name'] == away_team]['location.x'].median()

        # Update the DataFrame
        match_results_df.loc[match_results_df['match_id'] == match_id, [
            'ht_home_pressure_height', 'ht_away_pressure_height',
            'ft_home_pressure_height', 'ft_away_pressure_height'
        ]] = [
            ht_home_pressure, ht_away_pressure,
            ft_home_pressure, ft_away_pressure
        ]

    return match_results_df

add_pressure_height_stats(df_events, match_results)

In [ ]:
def add_progressive_passes_stats(events_df, match_results_df):
    for match_id in match_results_df['match_id']:
        match_events = events_df[events_df['match_id'] == match_id]
        home_team = match_results_df.loc[match_results_df['match_id'] == match_id, 'home_team'].iloc[0]
        away_team = match_results_df.loc[match_results_df['match_id'] == match_id, 'away_team'].iloc[0]

        # First time statistics
        ht_events = match_events[match_events['period'] == 1]

        # Completed passes
        ht_passes = ht_events[
            (ht_events['type.name'] == 'Pass') &
            (ht_events['pass.outcome.name'].isna())  # passaggi completati
        ]

        def is_progressive(row):
            start_dist = np.sqrt((120 - row['location.x'])**2 + (40 - row['location.y'])**2)
            end_dist = np.sqrt((120 - row['pass.end_location.x'])**2 + (40 - row['pass.end_location.y'])**2)
            return end_dist < start_dist * 0.8  # riduzione almeno del 20%

        # Count first time progressive passes
        ht_home_prog = len(ht_passes[
            (ht_passes['team.name'] == home_team) &
            (ht_passes.apply(is_progressive, axis=1))
        ])

        ht_away_prog = len(ht_passes[
            (ht_passes['team.name'] == away_team) &
            (ht_passes.apply(is_progressive, axis=1))
        ])

        # Full time
        ft_passes = match_events[
            (match_events['type.name'] == 'Pass') &
            (match_events['pass.outcome.name'].isna())
        ]

        ft_home_prog = len(ft_passes[
            (ft_passes['team.name'] == home_team) &
            (ft_passes.apply(is_progressive, axis=1))
        ])

        ft_away_prog = len(ft_passes[
            (ft_passes['team.name'] == away_team) &
            (ft_passes.apply(is_progressive, axis=1))
        ])

        # Update the DataFrame
        match_results_df.loc[match_results_df['match_id'] == match_id, [
            'ht_home_progressive_passes', 'ht_away_progressive_passes',
            'ft_home_progressive_passes', 'ft_away_progressive_passes'
        ]] = [
            ht_home_prog, ht_away_prog,
            ft_home_prog, ft_away_prog
        ]

    return match_results_df

add_progressive_passes_stats(df_events, match_results)

In [ ]:
match_results.columns

# Analysis of the relationship between first half and final result

## 1. Distribution of results and patterns of change

### Result stability
62% of matches maintain the same result (1/X/2) as the first half, highlighting a strong continuity between the two game periods. Particularly interesting is how this stability varies based on the partial result:
- Home wins maintained: 80% (strong ability to manage home advantage)
- Away wins maintained: 76% (similar resilience even away from home)
- Draws maintained: 42% (greater instability in tied situations)

### Comeback patterns
- Comebacks occur in 37% of matches, indicating that more than a third of matches see a change in result
- There is a clear home advantage in comebacks (18% vs 11%), probably due to crowd support and familiarity with the field

In [ ]:
# Basic statistics on result stability
result_stability = len(match_results[match_results['ht_result'] == match_results['ft_result']]) / len(match_results) * 100
# Percentage of matches where the home team led at HT and kept the lead at FT
home_wins_kept = len(match_results[(match_results['ht_result'] == '1') & (match_results['ft_result'] == '1')]) / len(match_results[match_results['ht_result'] == '1']) * 100
# Percentage of matches that ended in a draw at HT and remained a draw at FT
draws_kept = len(match_results[(match_results['ht_result'] == 'X') & (match_results['ft_result'] == 'X')]) / len(match_results[match_results['ht_result'] == 'X']) * 100
# Percentage of matches where the away team led at HT and kept the lead at FT
away_wins_kept = len(match_results[(match_results['ht_result'] == '2') & (match_results['ft_result'] == '2')]) / len(match_results[match_results['ht_result'] == '2']) * 100

# Statistics on comebacks
comebacks = len(match_results[match_results['ht_result'] != match_results['ft_result']]) / len(match_results) * 100
# Percentage of matches where the HT result was either a draw or an away lead, but the FT result was a home win
home_comebacks = len(match_results[(match_results['ht_result'].isin(['2', 'X'])) & (match_results['ft_result'] == '1')]) / len(match_results) * 100
# Percentage of matches where the HT result was either a draw or a home lead, but the FT result was an away win
away_comebacks = len(match_results[(match_results['ht_result'].isin(['1', 'X'])) & (match_results['ft_result'] == '2')]) / len(match_results) * 100

print(f"Result stability: {result_stability:.1f}%")
print(f"Home wins held: {home_wins_kept:.1f}%")
print(f"Draws held: {draws_kept:.1f}%")
print(f"Away wins held: {away_wins_kept:.1f}%")
print(f"\nTotal comebacks: {comebacks:.1f}%")
print(f"Home comebacks: {home_comebacks:.1f}%")
print(f"Away comebacks: {away_comebacks:.1f}%")


In [ ]:
# Create the figure
fig = plt.figure(figsize=(6,3), dpi=300, facecolor='blanchedalmond')
ax = plt.subplot(111)
ax.set_facecolor('blanchedalmond')

# Create transition matrix (in percentage) between HT and FT results
result_flow = pd.crosstab(match_results['ht_result'], match_results['ft_result'], normalize='index') * 100

print(result_flow)

# Prepare data for heatmap
data = result_flow.values
x_labels = ['Home Win', 'Away Win', 'Draw']
y_labels = ['Home Win', 'Away Win', 'Draw']

# Custom colormap similar to 'SOC4'
cmap = plt.cm.get_cmap('SOC4')

# Create heatmap using matplotlib
im = ax.imshow(data, cmap=cmap, vmin=0, vmax=100)

# Add numerical values to each cell
for i in range(len(y_labels)):
    for j in range(len(x_labels)):
        text = ax.text(j, i, f"{data[i, j]:.0f}%",
                       ha="center", va="center", color="white",
                       fontproperties=font_med, fontsize=6)

# Axis configuration
ax.set_xticks(np.arange(len(x_labels)))
ax.set_yticks(np.arange(len(y_labels)))
ax.set_xticklabels(x_labels, fontproperties=font_med, fontsize = 6)
ax.set_yticklabels(y_labels, fontproperties=font_med, fontsize = 6)

# Axis labels
ax.set_xlabel('Full-Time Result', fontproperties=font_med, color='black', size=8)
ax.set_ylabel('Half-Time Result', fontproperties=font_med, color='black', size=8)

# Colorbar
cbar = fig.colorbar(im, ax=ax)
cbar.ax.set_ylabel('Percentage %', fontproperties=font_med, size=8)
cbar.ax.tick_params(labelsize=6)

# Set font for colorbar ticks
for tick_label in cbar.ax.get_yticklabels():
    tick_label.set_fontproperties(font_med)

# Titles
fig_text(x=0.6, y=0.92,
         s="RESULT EVOLUTION FROM HALF-TIME TO FULL-TIME",
         va="bottom", ha="center",
         fontsize=12, font=font_semi, color="darkred")

fig_text(x=0.6, y=0.88,
         s="Percentage transition between results",
         va="bottom", ha="center",
         fontsize=8, font=font_med, color="darkorange")

# StatsBomb logo
logo = plt.imread('/content/drive/MyDrive/SB - Icon Lockup - Colour positive.png')
logo_ax = fig.add_axes([0.82, 0.02, 0.1, 0.02])
logo_ax.imshow(logo)
logo_ax.axis('off')

plt.tight_layout()
plt.savefig("matrixevo.png", bbox_inches='tight', dpi=300)
plt.show()


## 2. Analysis of metrics distribution by final result

The following **boxplots** show the distribution of metrics for each final result (win, draw, loss).


In [ ]:
# Creating the "result" column
match_results['result'] = match_results.apply(lambda row:
    1 if row['ft_home_goals'] > row['ft_away_goals'] else  # Home win
    -1 if row['ft_home_goals'] < row['ft_away_goals'] else  # Away win
    0, axis=1)  # Draw

# Selecting metrics to analyze
metrics_to_plot = [
    'xg', 'shots', 'shots_on_target', 'possession', 'passes_completed',
    'field_tilt', 'duels_won', 'interceptions_blocks', 'clearances',
    'ppda', 'pressure_height', 'progressive_passes'
]

# Creating aggregated datasets for each result type (Home Win, Draw, Away Win)
data = {'V': {}, 'X': {}, 'S': {}}

for metric in metrics_to_plot:
    # For home wins (V), take the home team's metric
    data['V'][metric] = match_results.apply(lambda row: row[f'ft_home_{metric}'] if row['result'] == 1
                                             else row[f'ft_away_{metric}'] if row['result'] == -1
                                             else np.nan, axis=1)
    # For away wins (S), take the away team's metric
    data['S'][metric] = match_results.apply(lambda row: row[f'ft_away_{metric}'] if row['result'] == 1
                                             else row[f'ft_home_{metric}'] if row['result'] == -1
                                             else np.nan, axis=1)
    # For draws (X), take the average of home and away team's metrics
    data['X'][metric] = match_results.apply(lambda row: np.mean([row[f'ft_home_{metric}'], row[f'ft_away_{metric}']])
                                             if row['result'] == 0 else np.nan, axis=1)

# Convert the dictionary into a DataFrame for easier plotting
boxplot_data = pd.DataFrame(data)

# Creating the figure with a grid of subplots
fig = plt.figure(figsize=(20, 13), dpi=300, facecolor='blanchedalmond')
fig.suptitle('DISTRIBUTION OF METRICS BY FINAL RESULT', fontsize=30, fontproperties=font_semi, color="darkred", y=0.95)

# Define the number of rows and columns for subplots
n_rows = 3
n_cols = 4

# Create the subplots for each metric
for idx, metric in enumerate(metrics_to_plot, 1):
    ax = plt.subplot(n_rows, n_cols, idx)

    # Set subplot background color
    ax.set_facecolor('blanchedalmond')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('black')
    ax.spines['left'].set_color('black')

    # Add horizontal gridlines to the plot
    ax.yaxis.grid(True, linestyle='--', alpha=0.3, color='black')
    ax.set_axisbelow(True)

    # Extract the values for each result category (Home Win, Draw, Away Win) for the boxplot
    boxplot_values = [boxplot_data['V'][metric].dropna(),
                     boxplot_data['X'][metric].dropna(),
                     boxplot_data['S'][metric].dropna()]

    # Create the boxplot at specified positions (1, 3, 5)
    positions = [1, 3, 5]
    bp = ax.boxplot(boxplot_values, positions=positions, widths=0.8, patch_artist=True)

    # Customize the boxplot's appearance (color, edges, etc.)
    for box in bp['boxes']:
        box.set_facecolor((0.545, 0, 0, 0.2))  # Semi-transparent red
        box.set_edgecolor('black')
        box.set_linewidth(1.5)
    for whisker in bp['whiskers']:
        whisker.set_color('black')
        whisker.set_linewidth(1.5)
    for cap in bp['caps']:
        cap.set_color('black')
        cap.set_linewidth(1.5)
    for median in bp['medians']:
        median.set_color('darkred')  # Red median line
        median.set_linewidth(2)

    # Add scatter points to visualize individual data points with slight jitter to avoid overlap
    for i, (key, color) in enumerate(zip(['V', 'X', 'S'], ['darkred', 'darkorange', 'black'])):
        jitter = np.random.uniform(-0.15, 0.15, size=len(boxplot_data[key][metric].dropna()))
        ax.scatter(np.full_like(boxplot_data[key][metric].dropna(), positions[i]) + jitter,
                  boxplot_data[key][metric].dropna(), color=color, alpha=0.6, s=20, edgecolors='black')

    # Set titles and labels for the subplot
    ax.set_title(f"{metric}", fontsize=12, fontweight='bold', color="darkred")
    ax.set_ylabel(metric, fontsize=10, color="black")

    # Modify the x-axis ticks and labels for the categories
    ax.set_xticks(positions)
    ax.set_xticklabels(['V', 'X', 'S'], fontproperties=font_semi, color='black', size=8)
    ax.tick_params(axis='y', colors='black', labelsize=8)

    # Hide the bottom spine and adjust x-limits to fit the positions
    ax.spines['bottom'].set_visible(False)
    ax.set_xlim(0, 6)

# Add the StatsBomb logo at the top right corner of the figure
logo = plt.imread('/content/drive/MyDrive/SB - Icon Lockup - Colour positive.png')
logo_ax = fig.add_axes([0.85, 0.9, 0.1, 0.09])
logo_ax.imshow(logo)
logo_ax.axis('off')

# Ensure that the layout fits within the figure boundaries and show the plot
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


# Modelling

## 1. Installation and library imports

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix,roc_curve, auc, roc_auc_score
from sklearn.inspection import permutation_importance
from scipy import stats
from sklearn.model_selection import cross_val_score, StratifiedKFold
from itertools import cycle

## 2. Data preparation
The features are prepared for the classification models, selecting statistics from the first half (prefix 'ht_') and from the entire match (prefix 'ft_').

In [ ]:
# Function to prepare features and targets
def prepare_features_targets(df):
    """
    Prepares the features and target for classification models.
    """
    # Create a copy to avoid modifying the original DataFrame
    df = df.copy()

    # Handle missing values (if any)
    if df.isnull().sum().sum() > 0:
        # Fill missing values with the column mean
        df = df.fillna(df.mean())

    # Select columns with the 'ht_' prefix as features for the first half
    ht_features = [col for col in df.columns if col.startswith('ht_') and col != 'ht_result']

    # Select columns with the 'ft_' prefix as features for the full match
    ft_features = [col for col in df.columns if col.startswith('ft_') and col != 'ft_result']

    # The target variable is the final match result ('ft_result')
    target = 'ft_result'

    # Create DataFrames for the features and target
    X_ht = df[ht_features]
    X_ft = df[ft_features]
    y = df[target]

    # Print the number of features for the first half and full match
    print(f"First half features: {len(ht_features)}")
    print(f"Full match features: {len(ft_features)}")

    return X_ht, X_ft, y

# Call the function to get the features and target
X_ht, X_ft, y = prepare_features_targets(match_results)


## 3. Testing the Null Hypothesis of Independence

To verify the null hypothesis of independence between the first half result and the final result, a chi-square test is performed. Additionally, Cramer's V coefficient is calculated to measure the strength of the association and a heatmap of the contingency table is displayed.

In [ ]:
# Function to perform chi-squared test for independence
def test_independence(df):
    """
    Performs the chi-squared test to check the null hypothesis of independence.
    """
    print("\n--- CHI-SQUARED TEST FOR THE NULL HYPOTHESIS ---")

    # Create a contingency table between first half result and final result
    ht_ft_contingency = pd.crosstab(df['ht_result'], df['ft_result'])
    print("\nContingency table between first half result and final result:")
    print(ht_ft_contingency)

    # Perform the chi-squared test
    chi2, p, dof, expected = stats.chi2_contingency(ht_ft_contingency)
    print(f"\nChi-squared test: chi2={chi2:.3f}, p-value={p:.6f}")
    print(f"Degrees of freedom: {dof}")

    # Decision based on the p-value
    if p < 0.05:
        print("Result: Reject the null hypothesis of independence.")
        print("There is a statistically significant relationship between the first half result and the final result.")
    else:
        print("Result: We cannot reject the null hypothesis of independence.")
        print("There is no statistically significant relationship between the first half result and the final result.")

    # Cramer's V coefficient for the strength of the association
    n = ht_ft_contingency.sum().sum()
    cramer_v = np.sqrt(chi2 / (n * (min(ht_ft_contingency.shape) - 1)))
    print(f"Cramer's V: {cramer_v:.3f}")
    print("Interpretation of Cramer's V:")
    print("- 0.0 to 0.1: very weak association")
    print("- 0.1 to 0.3: weak association")
    print("- 0.3 to 0.5: moderate association")
    print("- 0.5 to 0.8: strong association")
    print("- 0.8 to 1.0: very strong association")

    return cramer_v

# Perform the test using the match results dataset
cramer_v = test_independence(match_results)


<userStyle>Normal</userStyle>

## 4. Creation and Evaluation of Models

The classification models implemented are SVM, Decision Tree, and Naive Bayes, using both first half data and entire match data. Then, performances are compared and results are visualized.

For this study, three different classification algorithms were selected, each with complementary characteristics that allow a complete view of the predictive problem:

- **Support Vector Machine (SVM)**: Chosen for its ability to find optimal hyperplanes in multidimensional spaces. In the football context, where relationships between statistics and final result are complex and non-linear, SVM can identify subtle patterns thanks to its flexibility in defining decision boundaries.

- **Decision Tree**: Selected for its interpretable nature and ability to capture hierarchical relationships between variables. Decision trees naturally reflect human decision-making processes, creating easily understandable if-then rules. This allows us to identify which first half statistics are decisive for predicting the final result.

- **Naive Bayes**: Incorporated as a representative of probabilistic approaches and as a Bayesian baseline. Despite the simplifying assumption of conditional independence between features (rarely true in sports data), this model offers an interesting contrast with more complex approaches and helps identify when additional complexity is truly necessary.

In [ ]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

In [ ]:
def build_and_evaluate_models(X_ht, X_ft, y):
    """
    Builds and evaluates predictive models using cross-validation and ROC curve analysis.
    """
    print("\n--- BUILDING AND EVALUATING MODELS ---")

    # Split into training and test sets
    X_ht_train, X_ht_test, X_ft_train, X_ft_test, y_train, y_test = train_test_split(
        X_ht, X_ft, y, test_size=0.25, random_state=42, stratify=y
    )

    print(f"Training set size: {X_ht_train.shape[0]} matches")
    print(f"Test set size: {X_ht_test.shape[0]} matches")

    # Standardization
    scaler_ht = StandardScaler()
    scaler_ft = StandardScaler()

    X_ht_train_scaled = scaler_ht.fit_transform(X_ht_train)
    X_ht_test_scaled = scaler_ht.transform(X_ht_test)

    X_ft_train_scaled = scaler_ft.fit_transform(X_ft_train)
    X_ft_test_scaled = scaler_ft.transform(X_ft_test)

    # Prepare for ROC curve
    classes = sorted(y.unique())
    n_classes = len(classes)
    y_test_bin = label_binarize(y_test, classes=classes)

    models = {
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'SVM': SVC(random_state=42, probability=True),
        'Naive Bayes': GaussianNB()
    }

    # Evaluate models
    results = {}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Create figure for ROC curves
    plt.figure(figsize=(20, 10), dpi=300, facecolor='blanchedalmond')

    # Two subplots: one for first half, one for full match
    ax1 = plt.subplot(1, 2, 1)
    ax1.set_facecolor('blanchedalmond')
    ax2 = plt.subplot(1, 2, 2)
    ax2.set_facecolor('blanchedalmond')

    for name, model in models.items():
        print(f"\nEvaluating model: {name}")

        # ----- FIRST HALF -----
        print("\nModel with first half data:")

        # Cross-validation
        cv_scores = cross_val_score(model, X_ht_train_scaled, y_train, cv=cv, scoring='accuracy')
        print(f"Cross-validation accuracy: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

        # Training and evaluation
        model.fit(X_ht_train_scaled, y_train)
        y_ht_pred = model.predict(X_ht_test_scaled)
        ht_accuracy = accuracy_score(y_test, y_ht_pred)
        print(f"Test accuracy: {ht_accuracy:.4f}")
        print(classification_report(y_test, y_ht_pred))

        # ROC curve for first half
        try:
            y_ht_score = model.predict_proba(X_ht_test_scaled)

            # Calculate macro-average AUC
            ht_roc_auc = {}
            for i, class_name in enumerate(classes):
                fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_ht_score[:, i])
                ht_roc_auc[class_name] = auc(fpr, tpr)

                # Plot ROC curve for each class
                ax1.plot(fpr, tpr, lw=2,
                       label=f'{name} - {class_name} (AUC = {ht_roc_auc[class_name]:.2f})')

            print(f"ROC AUC (macro): {np.mean(list(ht_roc_auc.values())):.4f}")
        except:
            print(f"Could not calculate ROC curves for {name} (first half)")

        # ----- FULL MATCH -----
        print("\nModel with full match data:")

        # Cross-validation
        cv_scores = cross_val_score(model, X_ft_train_scaled, y_train, cv=cv, scoring='accuracy')
        print(f"Cross-validation accuracy: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

        # Training and evaluation
        model.fit(X_ft_train_scaled, y_train)
        y_ft_pred = model.predict(X_ft_test_scaled)
        ft_accuracy = accuracy_score(y_test, y_ft_pred)
        print(f"Test accuracy: {ft_accuracy:.4f}")
        print(classification_report(y_test, y_ft_pred))

        # ROC curve for full match
        try:
            y_ft_score = model.predict_proba(X_ft_test_scaled)

            # Calculate macro-average AUC
            ft_roc_auc = {}
            for i, class_name in enumerate(classes):
                fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_ft_score[:, i])
                ft_roc_auc[class_name] = auc(fpr, tpr)

                # Plot ROC curve for each class
                ax2.plot(fpr, tpr, lw=2,
                       label=f'{name} - {class_name} (AUC = {ft_roc_auc[class_name]:.2f})')

            print(f"ROC AUC (macro): {np.mean(list(ft_roc_auc.values())):.4f}")
        except:
            print(f"Could not calculate ROC curves for {name} (full match)")

        # Store results
        results[name] = {
            'ht_accuracy': ht_accuracy,
            'ft_accuracy': ft_accuracy,
            'improvement': ft_accuracy - ht_accuracy,
            'ht_model': model,
            'ht_predictions': y_ht_pred,
            'ft_predictions': y_ft_pred,
            'ht_cv_accuracy': cv_scores.mean(),
            'ht_roc_auc': ht_roc_auc if 'ht_roc_auc' in locals() else None,
            'ft_roc_auc': ft_roc_auc if 'ft_roc_auc' in locals() else None
        }

    # Final ROC curve configuration
    for ax, title in zip([ax1, ax2], ['ROC Curve (First Half)', 'ROC Curve (Full Match)']):
        # Diagonal line (random classifier)
        ax.plot([0, 1], [0, 1], 'k--', lw=2)

        # Axis settings
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel('False Positive Rate', fontproperties=font_semi, color='black', size=12)
        ax.set_ylabel('True Positive Rate', fontproperties=font_semi, color='black', size=12)
        ax.set_title(title, fontproperties=font_semi, color='darkred', size=14)

        # Grid and style
        ax.grid(axis='both', color='lightgrey', ls=':', alpha=0.6)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('black')
        ax.spines['bottom'].set_color('black')

        # Legend
        ax.legend(loc="lower right", prop=font_med, fontsize=8)

    # Main title
    plt.suptitle("ROC CURVES FOR RESULT CLASSIFICATION",
                 fontproperties=font_semi, color='darkred', size=20, y=0.98)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig('roc_curves.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Best model
    best_model_name = max(results, key=lambda x: results[x]['ht_accuracy'])
    best_model = results[best_model_name]['ht_model']
    print(f"\nThe best model based on first half data is {best_model_name}")

    # Create summary DataFrame
    models_df = pd.DataFrame({
        'Model': list(results.keys()),
        'CV Accuracy First Half': [results[model]['ht_cv_accuracy'] for model in results],
        'Accuracy First Half': [results[model]['ht_accuracy'] for model in results],
        'Accuracy Full Match': [results[model]['ft_accuracy'] for model in results],
        'Improvement': [results[model]['improvement'] for model in results]
    })

    models_df = models_df.sort_values('Accuracy First Half', ascending=False)

    # Accuracy comparison chart (similar code as before)
    fig = plt.figure(figsize=(12, 6), dpi=300, facecolor='blanchedalmond')
    ax = plt.subplot(111)
    ax.set_facecolor('blanchedalmond')

    # Define bar width and positions
    width = 0.25
    x = np.arange(len(models_df))

    # Create bars
    bar1 = ax.bar(x - width, models_df['CV Accuracy First Half'], width, label='CV First Half',
                color='darkblue', edgecolor='black', linewidth=1, alpha=0.7)
    bar2 = ax.bar(x, models_df['Accuracy First Half'], width, label='Test First Half',
                color='darkred', edgecolor='black', linewidth=1, alpha=0.8)
    bar3 = ax.bar(x + width, models_df['Accuracy Full Match'], width, label='Test Full Match',
                color='darkorange', edgecolor='black', linewidth=1)

    # Remove borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('black')
    ax.spines['bottom'].set_color('black')

    # Grid
    ax.grid(axis='y', color='lightgrey', ls=':', alpha=0.6)
    ax.set_axisbelow(True)

    # Annotations for improvement (only between Test First Half and Test Full Match)
    for i, model in enumerate(models_df['Model']):
        improvement = models_df.loc[models_df['Model']==model, 'Improvement'].values[0]
        text_ = ax.annotate(f"+{improvement:.3f}",
                        xy=(i + width/2, models_df.loc[models_df['Model']==model, 'Accuracy Full Match'].values[0]),
                        xytext=(0, 5),
                        textcoords='offset points',
                        ha='center', va='bottom',
                        color='black', fontsize=10, fontproperties=font_semi,
                        weight='bold')
        text_.set_path_effects([path_effects.Stroke(linewidth=1.5, foreground='white'), path_effects.Normal()])

    # Titles and labels
    fig_text(x=0.5, y=0.95,
            s="COMPARISON OF MODEL PERFORMANCE WITH CROSS-VALIDATION",
            va="bottom", ha="center",
            fontsize=20, font=font_semi, color="darkred")

    fig_text(x=0.5, y=0.91,
            s="Predictive accuracy with first half vs. full match statistics",
            va="bottom", ha="center",
            fontsize=12, font=font_med, color="darkorange")

    # Axis labels and x-axis ticks
    ax.set_ylabel('Accuracy', fontproperties=font_semi, color='black', size=12)
    ax.set_xticks(x)
    ax.set_xticklabels(models_df['Model'], fontproperties=font_med, size=12)
    ax.tick_params(axis='y', labelsize=12, colors='black')
    ax.set_ylim(0, 1.05)  # Leave space for annotations

    # Legend
    legend = ax.legend(loc='upper right', prop=font_med)
    legend.get_frame().set_facecolor('blanchedalmond')
    legend.get_frame().set_edgecolor('black')

    plt.tight_layout()
    plt.show()

    return best_model, X_ht_train_scaled, y_train, results, X_ht.columns, y_test

best_model, X_ht_train_scaled, y_train, results, feature_names, y_test = build_and_evaluate_models(X_ht, X_ft, y)


The analysis of the three classification models shows significant results in terms of both accuracy and discriminative capacity:

### Performance with first half data
- **SVM** emerges as the most effective model, with a cross-validation accuracy of 59.30% (±6.79%) and an accuracy on the test set of 54.74%. The AUC ROC value of 0.72 indicates a good ability to discriminate between classes.
- **Naive Bayes** achieves a cross-validation accuracy of 56.14% but drops to 48.42% on the test set. The AUC ROC of 0.69 suggests that, despite the lower accuracy, it maintains a decent discriminative capacity.
- **Decision Tree** shows the weakest results for first half data, with a test accuracy of 43.16% and an AUC ROC of 0.57, just above random classification.

### Performance with complete match data
- **Decision Tree** achieves exceptional performance with complete data: 98.60% in cross-validation and 96.84% on the test set, with an AUC ROC of 0.97. This demonstrates how the algorithm can identify clear patterns when all match information is available.
- **SVM** maintains good performance with an accuracy of 84.21% on the test set and an AUC ROC of 0.94.
- **Naive Bayes** shows the most limited improvement, reaching only 58.95% accuracy, suggesting that its independence assumptions limit the ability to exploit the additional information.

### Differences between Cross-Validation and test
Cross-validation reveals interesting aspects about the robustness of the models:
- Decision Tree shows the greatest discrepancy between CV (52.28%) and test (43.16%) on first half data, indicating a certain instability
- SVM and Naive Bayes show more contained differences, suggesting greater predictive stability

The ROC curves clearly visualize the performance gap between models based on the first half and those on the complete match, illustrating how second half information is determinant for accurate prediction of the final result.

## 5. Feature importance analysis

This function analyzes the feature importance for the best identified model, visualizing the most relevant ones.


In [ ]:
def analyze_feature_importance(best_model, feature_names, X_ht_train_scaled, y_train):
    """
    Analyzes the feature importance for the best model.
    """
    print("\n--- FEATURE IMPORTANCE ANALYSIS ---")

    # Identify the type of model
    model_type = best_model.__class__.__name__
    print(f"Analysis for model: {model_type}")

    # Create a figure consistent with the style of other plots
    fig = plt.figure(figsize=(12, 7), dpi=300, facecolor='blanchedalmond')
    ax = plt.subplot(111)
    ax.set_facecolor('blanchedalmond')

    # Decision Tree has native feature_importances_
    if hasattr(best_model, 'feature_importances_'):
        print("Using the model's native feature importance...")

        # Feature importance based on the model
        feature_importance = pd.DataFrame({
            'Feature': feature_names,
            'Importance': best_model.feature_importances_
        }).sort_values('Importance', ascending=False)

        importance_source = "Feature Importance"

    # For SVM and Naive Bayes, we use permutation importance
    else:
        print("Calculating feature importance through permutation importance...")
        perm_importance = permutation_importance(best_model, X_ht_train_scaled, y_train, n_repeats=10, random_state=42)

        feature_importance = pd.DataFrame({
            'Feature': feature_names,
            'Importance': perm_importance.importances_mean
        }).sort_values('Importance', ascending=False)

        importance_source = "Permutation Importance"

    # Select the top 10 features
    top_features = feature_importance.head(10)

    # Display the top 10 most important features
    bars = ax.barh(np.arange(len(top_features)),
                 top_features['Importance'],
                 height=0.7,
                 color='darkred',
                 edgecolor='black',
                 alpha=0.8)

    # Add numerical values to the bars
    for i, (value, feature) in enumerate(zip(top_features['Importance'], top_features['Feature'])):
        text_ = ax.annotate(
            f"{value:.3f}",
            xy=(value, i),
            xytext=(5, 0),
            textcoords='offset points',
            ha='left', va='center',
            color='black',
            fontproperties=font_med,
            fontsize=10
        )
        text_.set_path_effects([path_effects.Stroke(linewidth=1.5, foreground='white'), path_effects.Normal()])

    # Axis configuration
    ax.set_yticks(np.arange(len(top_features)))
    ax.set_yticklabels(top_features['Feature'], fontproperties=font_med)
    ax.invert_yaxis()  # Most important features at the top

    # Configure x-axis ticks
    ax.tick_params(axis='x', labelsize=10, colors='black')
    for tick in ax.get_xticklabels():
        tick.set_fontproperties(font_med)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('black')
    ax.spines['bottom'].set_color('black')

    # Grid
    ax.grid(axis='x', color='lightgrey', ls=':', alpha=0.6)
    ax.set_axisbelow(True)

    # Titles
    fig_text(x=0.5, y=0.93,
            s=f"FEATURE IMPORTANCE OF FIRST HALF ({model_type})",
            va="bottom", ha="center",
            fontsize=20, font=font_semi, color="darkred")

    # Subtitle with method information
    fig_text(x=0.5, y=0.89,
            s=f"Method: {importance_source}",
            va="bottom", ha="center",
            fontsize=12, font=font_med, color="darkorange")

    # Axis labels
    ax.set_xlabel('Importance', fontproperties=font_semi, color='black', size=12)

    plt.tight_layout()  # Adjust layout for titles
    plt.show()

    return feature_importance

feature_importance = analyze_feature_importance(best_model, feature_names, X_ht_train_scaled, y_train)




The feature importance analysis through permutation importance reveals an informative picture of the first half statistics that most influence the final match result.

The importance values obtained (maximum 0.107) might appear relatively modest, but they represent a substantial impact in the context of permutation importance. This method directly measures how much model performance degrades when the information contained in a specific variable is randomized, providing a conservative but robust estimate of the actual contribution of each feature.

### Hierarchy of predictive statistics

The results highlight a clear hierarchy of importance:

1. **Goals scored** (`ht_away_goals`: 0.107, `ht_home_goals`: 0.097) predictably dominate the ranking, with a combined importance that confirms how the first half score is a strong predictive signal of the final result.

2. **Quality of chances** (`ht_home_shots_on_target`: 0.037, `ht_away_xg`: 0.029) emerges as the second dimension of importance, demonstrating that it's not just scoring goals, but also the ability to create dangerous chances that predicts the final result.

3. **Physical and tactical dominance** (`ht_home_duels_won`: 0.026) reveals the importance of aspects not directly related to finishing, suggesting how controlling the game through won duels is a significant indicator of the match evolution.

4. **Defensive solidity** (`ht_away_interceptions_blocks`: 0.022, `ht_away_clearances`: 0.020) completes the picture, emphasizing how an away team's ability to withstand pressure is a predictive element of the final result.

The relatively balanced distribution of importance among different metrics represents a strength in the analysis: it suggests that the SVM model has identified complex and multi-dimensional patterns in the data, not limiting itself to exploiting one or two dominant variables.

A value of 0.107 for the most important feature indicates that, when this information is compromised, the model loses over 10% of its predictive capacity - a substantial impact considering that it's a single variable among the many analyzed.

This distribution of importance reflects the multi-factorial nature of football, where the result is determined by a complex set of factors that interact with each other, and confirms our SVM model's ability to capture this complexity through the identification of non-linear patterns in the data.

## 6. Analysis of differences between first half and complete match




In [ ]:
# Analyzes the differences between the first half and full match result
def analyze_differences(df, results, y_test):
    """
    Analyzes the differences between the first half and the full match.
    """
    print("\n--- ANALYSIS OF THE DIFFERENCES BETWEEN FIRST HALF AND FULL MATCH RESULT ---")

    # Percentage of matches where the result stayed the same
    same_result = (df['ht_result'] == df['ft_result']).mean() * 100
    print(f"Percentage of matches where the first half result is the same as the full match result: {same_result:.2f}%")

    # Predictive improvement analysis
    print("\nPredictive improvement between first half and full match:")
    for model_name, model_data in results.items():
        ht_incorrect = (model_data['ht_predictions'] != y_test).sum()
        ft_incorrect = (model_data['ft_predictions'] != y_test).sum()
        improvement = ht_incorrect - ft_incorrect

        print(f"\n{model_name}:")
        print(f"Incorrect predictions (First Half): {ht_incorrect} ({ht_incorrect/len(y_test)*100:.2f}%)")
        print(f"Incorrect predictions (Full Match): {ft_incorrect} ({ft_incorrect/len(y_test)*100:.2f}%)")
        print(f"Improvement: {improvement} matches ({improvement/len(y_test)*100:.2f}%)")

    # Average improvement
    improvements = {model: results[model]['improvement'] * 100 for model in results}
    avg_improvement = sum(improvements.values()) / len(improvements)

    print(f"\nAverage improvement in accuracy using the full match data: {avg_improvement:.2f}%")

    return same_result, avg_improvement

# Run the analysis of differences
same_result_pct, avg_improvement = analyze_differences(match_results, results, y_test)

# Print the conclusions for the analysis with all statistics
print("\n\n============ CONCLUSIONS ============")
print(f"Cramer's V index between first half and final result: {cramer_v:.3f}")

if cramer_v < 0.3:
    association_strength = "weak"
elif cramer_v < 0.5:
    association_strength = "moderate"
else:
    association_strength = "strong"

print(f"The strength of the association between the first half and the final result is {association_strength}.")
print(f"{same_result_pct:.2f}% of matches end with the same result as the first half.")
print(f"The models improve on average by {avg_improvement:.2f}% when using the full match data.")

top_features = feature_importance['Feature'].head(5).tolist()
print(f"\nThe top 5 most important features of the first half for predicting the final result are:")
for i, feature in enumerate(top_features, 1):
    print(f"{i}. {feature}")

print("\nComparison with the null hypothesis:")
if cramer_v < 0.1:
    print("The results support the null hypothesis: there is a very weak relationship between the first half and the final result.")
else:
    print("The results do not fully support the null hypothesis: there is a relationship between the first half and the final result.")
    print(f"The strength of this relationship is {association_strength}, meaning there is a relationship but it is not deterministic, implying other factors such as fatigue, substitutions, tactical decisions, etc. are also involved.")


## 7. Data Preparation excluding goal statistics

The result of a match is inevitably influenced primarily by goals, both logically and from the analysis results, goals scored are the most important features.

Therefore, this function prepares the features excluding those related to goals and shots, to identify which other aspects of the game can be predictive of the final result.

In [ ]:
# Prepares the features excluding goal-related statistics
def prepare_features_targets_no_goals(df):
    """
    Prepares the features excluding variables related to goals.
    """
    # Create a copy for safety
    df = df.copy()

    # Handle missing values
    if df.isnull().sum().sum() > 0:
        df = df.fillna(df.mean())

    # Select columns excluding goal-related ones
    goal_related = ['goals']
    ht_features = [col for col in df.columns if col.startswith('ht_') and
                   col != 'ht_result' and
                   not any(term in col for term in goal_related)]

    ft_features = [col for col in df.columns if col.startswith('ft_') and
                   col != 'ft_result' and
                   not any(term in col for term in goal_related)]

    # The target variable is the final result
    target = 'ft_result'

    # Create the dataframes
    X_ht = df[ht_features]
    X_ft = df[ft_features]
    y = df[target]

    print("First half features (excluding goals and shots):", len(ht_features))
    print("Full match features (excluding goals and shots):", len(ft_features))
    print("\nFeatures included in the model:")
    for feature in ht_features:
        print(f"- {feature}")

    return X_ht, X_ft, y

X_ht_no_goals, X_ft_no_goals, y_no_goals = prepare_features_targets_no_goals(match_results)

## 8. Creation and evaluation of models without goals statistics


In [ ]:
def build_and_evaluate_models_no_goals(X_ht, X_ft, y):
    """
    Builds and evaluates predictive models without goal statistics,
    using cross-validation and ROC curves.
    """
    print("\n--- BUILDING AND EVALUATING MODELS (WITHOUT GOALS) ---")

    # Split into training and test sets
    X_ht_train, X_ht_test, X_ft_train, X_ft_test, y_train, y_test = train_test_split(
        X_ht, X_ft, y, test_size=0.25, random_state=42, stratify=y
    )

    print(f"Training set size: {X_ht_train.shape[0]} matches")
    print(f"Test set size: {X_ht_test.shape[0]} matches")

    # Standardization
    scaler_ht = StandardScaler()
    scaler_ft = StandardScaler()

    X_ht_train_scaled = scaler_ht.fit_transform(X_ht_train)
    X_ht_test_scaled = scaler_ht.transform(X_ht_test)

    X_ft_train_scaled = scaler_ft.fit_transform(X_ft_train)
    X_ft_test_scaled = scaler_ft.transform(X_ft_test)

    # Prepare for ROC curve
    classes = sorted(y.unique())
    n_classes = len(classes)
    y_test_bin = label_binarize(y_test, classes=classes)

    # Models to test (replaced with new classifiers)
    models = {
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'SVM': SVC(random_state=42, probability=True),
        'Naive Bayes': GaussianNB()
    }

    # Model evaluation
    results_no_goals = {}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Create figure for ROC curves
    plt.figure(figsize=(20, 10), dpi=300, facecolor='blanchedalmond')

    # Two subplots: one for first half, one for full match
    ax1 = plt.subplot(1, 2, 1)
    ax1.set_facecolor('blanchedalmond')
    ax2 = plt.subplot(1, 2, 2)
    ax2.set_facecolor('blanchedalmond')

    # Iterate over models for evaluation
    for name, model in models.items():
        print(f"\nEvaluating model: {name}")

        # ----- FIRST HALF (WITHOUT GOALS) -----
        print("Model with first-half data (without goals):")

        # Cross-validation
        cv_scores = cross_val_score(model, X_ht_train_scaled, y_train, cv=cv, scoring='accuracy')
        print(f"Cross-validation accuracy: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

        # Training and evaluation
        model.fit(X_ht_train_scaled, y_train)
        y_ht_pred = model.predict(X_ht_test_scaled)
        ht_accuracy = accuracy_score(y_test, y_ht_pred)
        print(f"Test accuracy: {ht_accuracy:.4f}")
        print(classification_report(y_test, y_ht_pred))

        # ROC curve for first half
        try:
            y_ht_score = model.predict_proba(X_ht_test_scaled)

            # Calculate macro-average AUC
            ht_roc_auc = {}
            for i, class_name in enumerate(classes):
                fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_ht_score[:, i])
                ht_roc_auc[class_name] = auc(fpr, tpr)

                # Plot ROC curve for each class
                ax1.plot(fpr, tpr, lw=2,
                       label=f'{name} - {class_name} (AUC = {ht_roc_auc[class_name]:.2f})')

            print(f"ROC AUC (macro): {np.mean(list(ht_roc_auc.values())):.4f}")
        except:
            print(f"Could not calculate ROC curves for {name} (first half)")

        # ----- FULL MATCH (WITHOUT GOALS) -----
        print("Model with full match data (without goals):")

        # Cross-validation
        cv_scores = cross_val_score(model, X_ft_train_scaled, y_train, cv=cv, scoring='accuracy')
        print(f"Cross-validation accuracy: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

        # Training and evaluation
        model.fit(X_ft_train_scaled, y_train)
        y_ft_pred = model.predict(X_ft_test_scaled)
        ft_accuracy = accuracy_score(y_test, y_ft_pred)
        print(f"Test accuracy: {ft_accuracy:.4f}")
        print(classification_report(y_test, y_ft_pred))

        # ROC curve for full match
        try:
            y_ft_score = model.predict_proba(X_ft_test_scaled)

            # Calculate macro-average AUC
            ft_roc_auc = {}
            for i, class_name in enumerate(classes):
                fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_ft_score[:, i])
                ft_roc_auc[class_name] = auc(fpr, tpr)

                # Plot ROC curve for each class
                ax2.plot(fpr, tpr, lw=2,
                       label=f'{name} - {class_name} (AUC = {ft_roc_auc[class_name]:.2f})')

            print(f"ROC AUC (macro): {np.mean(list(ft_roc_auc.values())):.4f}")
        except:
            print(f"Could not calculate ROC curves for {name} (full match)")

        # Store results
        results_no_goals[name] = {
            'ht_accuracy': ht_accuracy,
            'ft_accuracy': ft_accuracy,
            'improvement': ft_accuracy - ht_accuracy,
            'ht_model': model,
            'ht_predictions': y_ht_pred,
            'ft_predictions': y_ft_pred,
            'ht_cv_accuracy': cv_scores.mean(),
            'ht_roc_auc': ht_roc_auc if 'ht_roc_auc' in locals() else None,
            'ft_roc_auc': ft_roc_auc if 'ft_roc_auc' in locals() else None
        }

    # Final configuration for ROC curves
    for ax, title in zip([ax1, ax2], ['ROC Curve (First Half - Without Goals)', 'ROC Curve (Full Match - Without Goals)']):
        # Diagonal line (random classifier)
        ax.plot([0, 1], [0, 1], 'k--', lw=2)

        # Axis settings
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel('False Positive Rate', fontproperties=font_semi, color='black', size=12)
        ax.set_ylabel('True Positive Rate', fontproperties=font_semi, color='black', size=12)
        ax.set_title(title, fontproperties=font_semi, color='darkred', size=14)

        # Grid and style
        ax.grid(axis='both', color='lightgrey', ls=':', alpha=0.6)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('black')
        ax.spines['bottom'].set_color('black')

        # Legend
        ax.legend(loc="lower right", prop=font_med, fontsize=8)

    # Main title
    plt.suptitle("ROC CURVES FOR MATCH RESULT CLASSIFICATION (WITHOUT GOALS)",
                 fontproperties=font_semi, color='darkred', size=20, y=0.98)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig('roc_curves_no_goals.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Best model
    best_model_name = max(results_no_goals, key=lambda x: results_no_goals[x]['ht_accuracy'])
    best_model = results_no_goals[best_model_name]['ht_model']
    print(f"\nThe best model based on first-half data (without goals) is {best_model_name}")

    # Create summary DataFrame
    models_df = pd.DataFrame({
        'Model': list(results_no_goals.keys()),
        'CV Accuracy First Half': [results_no_goals[model]['ht_cv_accuracy'] for model in results_no_goals],
        'Accuracy First Half': [results_no_goals[model]['ht_accuracy'] for model in results_no_goals],
        'Accuracy Full Match': [results_no_goals[model]['ft_accuracy'] for model in results_no_goals],
        'Improvement': [results_no_goals[model]['improvement'] for model in results_no_goals]
    })

    models_df = models_df.sort_values('Accuracy First Half', ascending=False)

    # Bar plot for model comparison
    fig = plt.figure(figsize=(12, 8), dpi=300, facecolor='blanchedalmond')
    ax = plt.subplot(111)
    ax.set_facecolor('blanchedalmond')

    # Define bar width and positions
    width = 0.25
    x = np.arange(len(models_df))

    # Create bars
    bar1 = ax.bar(x - width, models_df['CV Accuracy First Half'], width, label='CV First Half',
                color='darkblue', edgecolor='black', linewidth=1, alpha=0.7)
    bar2 = ax.bar(x, models_df['Accuracy First Half'], width, label='Test First Half',
                color='darkred', edgecolor='black', linewidth=1, alpha=0.8)
    bar3 = ax.bar(x + width, models_df['Accuracy Full Match'], width, label='Test Full Match',
                color='darkorange', edgecolor='black', linewidth=1)

    # Remove top and right borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('black')
    ax.spines['bottom'].set_color('black')

    # Grid
    ax.grid(axis='y', color='lightgrey', ls=':', alpha=0.6)
    ax.set_axisbelow(True)

    # Annotations for improvement
    for i, model in enumerate(models_df['Model']):
        improvement = models_df.loc[models_df['Model']==model, 'Improvement'].values[0]
        text_ = ax.annotate(f"+{improvement:.3f}",
                        xy=(i + width/2, models_df.loc[models_df['Model']==model, 'Accuracy Full Match'].values[0]),
                        xytext=(0, 5),
                        textcoords='offset points',
                        ha='center', va='bottom',
                        color='black', fontsize=10, fontproperties=font_semi,
                        weight='bold')
        text_.set_path_effects([path_effects.Stroke(linewidth=1.5, foreground='white'), path_effects.Normal()])

    # Titles and labels
    fig_text(x=0.5, y=0.95,
            s="MODEL PERFORMANCE COMPARISON (WITHOUT GOALS) WITH CROSS-VALIDATION",
            va="bottom", ha="center",
            fontsize=20, font=font_semi, color="darkred")

    fig_text(x=0.5, y=0.91,
            s="Predictive accuracy with non-goal-related statistics",
            va="bottom", ha="center",
            fontsize=12, font=font_med, color="darkorange")

    # Axis labels and x-axis tick labels
    ax.set_ylabel('Accuracy', fontproperties=font_semi, color='black', size=12)
    ax.set_xticks(x)
    ax.set_xticklabels(models_df['Model'], fontproperties=font_med, size=12)
    ax.tick_params(axis='y', labelsize=12, colors='black')
    ax.set_ylim(0, 1.05)  # Leave space for annotations

    # Legend
    legend = ax.legend(loc='upper right', prop=font_med)
    legend.get_frame().set_facecolor('blanchedalmond')
    legend.get_frame().set_edgecolor('black')

    plt.tight_layout()
    plt.savefig('model_comparison_no_goals_with_cv.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Return the best model, training data, and results
    return best_model, X_ht_train_scaled, y_train, results_no_goals, X_ht.columns, y_test

# Call the function with your data
best_model_no_goals, X_ht_train_scaled_no_goals, y_train_no_goals, results_no_goals, feature_names_no_goals, y_test_no_goals = build_and_evaluate_models_no_goals(X_ht_no_goals, X_ft_no_goals, y_no_goals)


The analysis of models without goal-related statistics offers valuable insights into the predictive capacity of other game metrics:

### Performance with first half data
- **SVM** still emerges as the most robust model in cross-validation (56.14% ±5.55%), despite the accuracy on the test set dropping to 47.37%. The AUC ROC value of 0.64 indicates a decent discriminative capacity.
- **Naive Bayes** shows good stability between CV (51.93%) and test (45.26%), with an AUC of 0.63 similar to SVM, suggesting that probabilistic relationships without goals maintain predictive value.
- **Decision Tree** obtains results comparable to other models on the test set (43.16%), but with the lowest AUC (0.56), indicating a lower ability to correctly distinguish between classes.

### Performance with complete match data
- **SVM** achieves the best performance with complete data (63.16% on the test), with a significant AUC of 0.76, confirming its ability to capture complex relationships between variables.
- **Decision Tree** and **Naive Bayes** show more modest improvements (50.53% and 49.47% respectively), with AUC around 0.61-0.62.

### Observations on ROC Curves
The ROC curves reveal interesting patterns:
- In the first half, the curves remain relatively close to the diagonal (random classification)
- With complete data, some curves (especially SVM's) move more decisively away from the diagonal, indicating greater discriminative power
- The difference between first half and complete match curves is less dramatic compared to the analysis with goals

### Comparison between CV and Test
A notable aspect is the discrepancy between CV and test accuracy, especially for SVM (about 9 percentage points). This suggests:
- Some variability in the data
- Possible overfitting on some characteristics of the training data
- The intrinsic difficulty of generalizing predictive patterns when the most important metrics (goals) are absent

The improvements between first half and complete match are more modest compared to the analysis with goals (+15.8% for SVM, +7.4% for Decision Tree, +4.2% for Naive Bayes), confirming that second half events have less predictive power when goals are excluded from the analysis.

These results demonstrate that, even in the absence of information on goals, there are statistical patterns in first half data that offer a limited but significant predictive value on the final result of matches.

## 9. Feature importance analysis without goal statistics

In [ ]:
def analyze_feature_importance_no_goals(best_model, feature_names, X_ht_train_scaled, y_train):
    """
    Analyzes feature importance for the model without goals.
    """
    print("\n--- FEATURE IMPORTANCE ANALYSIS (NO GOALS) ---")

    # Identify the model type
    model_type = best_model.__class__.__name__
    print(f"Analysis for model: {model_type}")

    # Create a figure with consistent style
    fig = plt.figure(figsize=(12, 7), dpi=300, facecolor='blanchedalmond')
    ax = plt.subplot(111)
    ax.set_facecolor('blanchedalmond')

    # Decision Tree has native feature_importances_
    if hasattr(best_model, 'feature_importances_'):
        print("Using native model feature importances...")

        # Feature importance based on the model
        feature_importance = pd.DataFrame({
            'Feature': feature_names,
            'Importance': best_model.feature_importances_
        }).sort_values('Importance', ascending=False)

        importance_source = "Feature Importance"

    # For SVM and Naive Bayes we use permutation importance
    else:
        print("Calculating feature importance using permutation importance...")
        perm_importance = permutation_importance(best_model, X_ht_train_scaled, y_train, n_repeats=10, random_state=42)

        feature_importance = pd.DataFrame({
            'Feature': feature_names,
            'Importance': perm_importance.importances_mean
        }).sort_values('Importance', ascending=False)

        importance_source = "Permutation Importance"

    # Select only the top 10 features
    top_features = feature_importance.head(10)

    # Display the most important features with customized style
    bars = ax.barh(np.arange(len(top_features)),
                 top_features['Importance'],
                 height=0.7,
                 color='darkred',
                 edgecolor='black',
                 alpha=0.8)

    # Add numeric values to the bars
    for i, (value, feature) in enumerate(zip(top_features['Importance'], top_features['Feature'])):
        text_ = ax.annotate(
            f"{value:.3f}",
            xy=(value, i),
            xytext=(5, 0),
            textcoords='offset points',
            ha='left', va='center',
            color='black',
            fontproperties=font_med,
            fontsize=10
        )
        text_.set_path_effects([path_effects.Stroke(linewidth=1.5, foreground='white'), path_effects.Normal()])

    # Configure axes
    ax.set_yticks(np.arange(len(top_features)))
    ax.set_yticklabels(top_features['Feature'], fontproperties=font_med)
    ax.invert_yaxis()  # Most important features at the top

    # Configure x-axis ticks
    ax.tick_params(axis='x', labelsize=10, colors='black')
    for tick in ax.get_xticklabels():
        tick.set_fontproperties(font_med)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('black')
    ax.spines['bottom'].set_color('black')

    # Grid
    ax.grid(axis='x', color='lightgrey', ls=':', alpha=0.6)
    ax.set_axisbelow(True)

    # Titles
    fig_text(x=0.5, y=0.93,
            s=f"FEATURE IMPORTANCE (NO GOALS) - {model_type}",
            va="bottom", ha="center",
            fontsize=20, font=font_semi, color="darkred")

    # Subtitle with method information
    fig_text(x=0.5, y=0.89,
            s=f"Method: {importance_source}",
            va="bottom", ha="center",
            fontsize=12, font=font_med, color="darkorange")

    # Axis labels
    ax.set_xlabel('Importance', fontproperties=font_semi, color='black', size=12)

    plt.tight_layout()
    plt.show()

    return feature_importance


feature_importance_no_goals = analyze_feature_importance_no_goals(best_model_no_goals, feature_names_no_goals, X_ht_train_scaled_no_goals, y_train_no_goals)


The importance values obtained (maximum 0.045) appear modest but still represent a relevant impact in the context of permutation importance, directly measuring how much model performance degrades when information from a specific variable is randomized.

### Hierarchy of predictive statistics

In the absence of goal statistics, a clear hierarchy of importance emerges:

1. **Expected Goals** (`ht_away_xg`: 0.045, `ht_home_xg`: 0.036) take the dominant position, confirming themselves as the best substitutes for actual goals. The away team's xG proves particularly predictive, suggesting that the quality of chances created by visitors has a superior signaling value.

2. **Shots on Target** (`ht_home_shots_on_target`: 0.040) emerge as the second most important metric, highlighting how the offensive precision of the home team in the first half is strongly correlated with the final result.

3. **Duels Won and Game Progression** (`ht_home_duels_won`: 0.020, `ht_away_progressive_passes`: 0.020) reveal the importance of tactical aspects and physical control of the match, highlighting how territorial dominance and the ability to advance the ball influence the final result.

4. **Volume of Shots and Passes** (`ht_home_shots`: 0.009, `ht_away_passes_completed`: 0.008) show a more modest but still significant importance, suggesting that the quantity of offensive actions and possession control have a complementary predictive value.

### Significance of results

These importance patterns reveal fundamental aspects of the game:

1. The quality of created chances (xG) is much more predictive than the quantity of shots or simple ball possession, confirming that offensive efficiency is more important than volume.

2. The away team's performance (xG, progressive passes) seems to have a particularly strong signaling value, suggesting that when a visiting team manages to generate quality chances in the first half, this is a strong indicator of the final result.

3. Duels won by the home team highlight the importance of the home field advantage and physical control of the match as a predictive element of the final result.

This distribution of importance reflects the multi-dimensional nature of football, where in the absence of goal metrics, other indicators of offensive quality, progression, and control take on a primary predictive role, confirming that the SVM model is able to identify complex relationships between these variables and the final result.

## 10. Comparison with original models

At this point we compare the performance of models without goals with those of models using all statistics. Additionally, the average accuracy decrease when excluding goal-related statistics is visualized in a comparative graph.

In [ ]:
def compare_with_original_models(results_no_goals, original_results):
    """
    Compares the performance of the models without goals with the original ones.
    """
    print("\n--- COMPARISON WITH ORIGINAL MODELS ---")

    # Check which models are present in both results
    common_models = set(results_no_goals.keys()).intersection(set(original_results.keys()))

    if not common_models:
        print("Warning: No common models found between the two results sets.")
        print("Ensure you are using the same model names in both analyses.")
        return 0, 0

    print(f"Common models found for comparison: {', '.join(common_models)}")

    # Prepare data for comparison
    comparison_data = []
    for model_name in common_models:
        comparison_data.append({
            'Model': model_name,
            'Type': 'With Goals - First Half',
            'Accuracy': original_results[model_name]['ht_accuracy']
        })
        comparison_data.append({
            'Model': model_name,
            'Type': 'With Goals - Full Match',
            'Accuracy': original_results[model_name]['ft_accuracy']
        })
        comparison_data.append({
            'Model': model_name,
            'Type': 'Without Goals - First Half',
            'Accuracy': results_no_goals[model_name]['ht_accuracy']
        })
        comparison_data.append({
            'Model': model_name,
            'Type': 'Without Goals - Full Match',
            'Accuracy': results_no_goals[model_name]['ft_accuracy']
        })

    comparison_df = pd.DataFrame(comparison_data)

    # Create figure with consistent style
    fig = plt.figure(figsize=(14, 8), dpi=300, facecolor='blanchedalmond')
    ax = plt.subplot(111)
    ax.set_facecolor('blanchedalmond')

    # Define colors for each model type (more contrasting palette)
    color_map = {
        'With Goals - First Half': '#8B0000',  # Dark red
        'With Goals - Full Match': '#FF4500',  # Orange red
        'Without Goals - First Half': '#006400',  # Dark green
        'Without Goals - Full Match': '#32CD32'  # Lime green
    }

    # Get unique models and types for bar positioning
    models = comparison_df['Model'].unique()
    types = ['With Goals - First Half', 'With Goals - Full Match',
             'Without Goals - First Half', 'Without Goals - Full Match']

    # Bar width and positions
    bar_width = 0.2
    positions = np.arange(len(models))

    # Draw bars for each model type
    for i, tipo in enumerate(types):
        data_subset = comparison_df[comparison_df['Type'] == tipo]
        offset = (i - 1.5) * bar_width

        # Sort data to maintain the same order of models
        data_ordered = pd.DataFrame()
        for model in models:
            model_data = data_subset[data_subset['Model'] == model]
            if not model_data.empty:
                data_ordered = pd.concat([data_ordered, model_data])

        if not data_ordered.empty:
            bars = ax.bar(positions + offset, data_ordered['Accuracy'],
                        width=bar_width,
                        label=tipo,
                        color=color_map[tipo],
                        edgecolor='black',
                        linewidth=1)

    # Remove borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('black')
    ax.spines['bottom'].set_color('black')

    # Grid
    ax.grid(axis='y', color='lightgrey', ls=':', alpha=0.6)
    ax.set_axisbelow(True)

    # Titles
    fig_text(x=0.5, y=0.95,
            s="MODEL COMPARISON: WITH GOALS VS WITHOUT GOALS",
            va="bottom", ha="center",
            fontsize=20, font=font_semi, color="darkred")

    fig_text(x=0.5, y=0.91,
            s="Effect of excluding goal-related statistics on predictive ability",
            va="bottom", ha="center",
            fontsize=15, font=font_med, color="darkorange")

    # Axis labels
    ax.set_ylabel('Accuracy', fontproperties=font_semi, color='black', size=12)
    ax.set_ylim(0, 1.05)

    # Configure y-axis ticks
    ax.tick_params(axis='y', labelsize=10, colors='black')
    for tick in ax.get_yticklabels():
        tick.set_fontproperties(font_med)

    # Configure x-axis ticks
    ax.set_xticks(positions)
    ax.set_xticklabels(models, fontproperties=font_med, size=12)

    # Legend
    legend = ax.legend(prop=font_med, loc='upper right')
    legend.get_frame().set_facecolor('blanchedalmond')
    legend.get_frame().set_edgecolor('black')

    plt.tight_layout()
    plt.show()

    # Calculate the average performance drop for common models
    performance_drop_ht = {
        model: original_results[model]['ht_accuracy'] - results_no_goals[model]['ht_accuracy']
        for model in common_models
    }

    performance_drop_ft = {
        model: original_results[model]['ft_accuracy'] - results_no_goals[model]['ft_accuracy']
        for model in common_models
    }

    avg_drop_ht = sum(performance_drop_ht.values()) / len(performance_drop_ht) if performance_drop_ht else 0
    avg_drop_ft = sum(performance_drop_ft.values()) / len(performance_drop_ft) if performance_drop_ft else 0

    print(f"Average accuracy drop (First Half): {avg_drop_ht:.4f} ({avg_drop_ht*100:.2f}%)")
    print(f"Average accuracy drop (Full Match): {avg_drop_ft:.4f} ({avg_drop_ft*100:.2f}%)")

    return avg_drop_ht, avg_drop_ft


avg_drop_ht, avg_drop_ft = compare_with_original_models(results_no_goals, results)


## 11. Riepilogo finale

In [ ]:
# Final summary
print("\n\n============= FINAL SUMMARY =============")

# Extract top 5 features from feature importance dataframes
top_features = feature_importance['Feature'].head(5).tolist()
top_features_no_goals = feature_importance_no_goals['Feature'].head(5).tolist()

# Section 1: Results with all statistics included
print("1. With all statistics:")
print(f" - Association between halftime and full-time result: {association_strength} (Cramer's V = {cramer_v:.3f})")
print(f" - Percentage of matches with the same result: {same_result_pct:.2f}%")
print(f" - Average improvement with full-match data: {avg_improvement:.2f}%")

# Best model (halftime)
best_model_name = max(results, key=lambda x: results[x]['ht_accuracy'])
print(f" - Best model (halftime): {best_model_name} (Accuracy: {results[best_model_name]['ht_accuracy']:.2f})")
if 'ht_roc_auc' in results[best_model_name] and results[best_model_name]['ht_roc_auc']:
    print(f"   AUC ROC: {np.mean(list(results[best_model_name]['ht_roc_auc'].values())):.2f}")

# Best model (full-time)
print(f" - Best model (full match): {max(results, key=lambda x: results[x]['ft_accuracy'])} (Accuracy: {results[max(results, key=lambda x: results[x]['ft_accuracy'])]['ft_accuracy']:.2f})")

# Top 5 features with goals included
print(" - Top 5 features (with goal stats):")
for i, feature in enumerate(top_features, 1):
    print(f"   {i}. {feature}")

# Section 2: Results without goal-related statistics
print("\n2. Without goal-related statistics:")
print(f" - Accuracy drop (halftime): {avg_drop_ht*100:.2f}%")
print(f" - Accuracy drop (full match): {avg_drop_ft*100:.2f}%")

# Best model without goal-related stats
best_model_no_goals_name = max(results_no_goals, key=lambda x: results_no_goals[x]['ht_accuracy'])
print(f" - Best model (halftime): {best_model_no_goals_name} (Accuracy: {results_no_goals[best_model_no_goals_name]['ht_accuracy']:.2f})")
print(f" - Best model (full match): {max(results_no_goals, key=lambda x: results_no_goals[x]['ft_accuracy'])} (Accuracy: {results_no_goals[max(results_no_goals, key=lambda x: results_no_goals[x]['ft_accuracy'])]['ft_accuracy']:.2f})")

# Top 5 features without goals
print(" - Top 5 features (without goal stats):")
for i, feature in enumerate(top_features_no_goals, 1):
    print(f"   {i}. {feature}")

# Section 3: Cross-validation comparison
print("\n3. Comparison with Cross-Validation:")
if all('ht_cv_accuracy' in results[model] for model in results):
    print(f" - Average CV accuracy (halftime): {np.mean([results[model]['ht_cv_accuracy'] for model in results]):.2f}")
    print(f" - Average difference between CV and test (halftime): {np.mean([results[model]['ht_cv_accuracy'] - results[model]['ht_accuracy'] for model in results]):.2f}")

if all('ht_cv_accuracy' in results_no_goals[model] for model in results_no_goals):
    print(f" - CV accuracy average (without goal stats): {np.mean([results_no_goals[model]['ht_cv_accuracy'] for model in results_no_goals]):.2f}")

# Overall interpretation
print("\nOverall findings:")
if cramer_v < 0.1:
    print("The results support the null hypothesis: the relationship between halftime and full-time result is very weak.")
else:
    print("The results do not support the null hypothesis: there is a relationship between halftime and full-time result.")
    print(f"The strength of this relationship is {association_strength} (Cramer's V = {cramer_v:.3f}).")

print(f"The predictive models confirm this conclusion, showing an average improvement of {avg_improvement:.2f}% when using full-match data.")
print(f"Excluding goal-related stats reduces accuracy by {avg_drop_ht*100:.2f}% at halftime and by {avg_drop_ft*100:.2f}% for the full match.")
print("This shows that, while goals are the most predictive factor, other stats (like xG, shots on target, possession) still retain strong predictive value.")

# Practical implications
print("\nImplications for football analysis:")
print("1. Halftime results are a moderate but significant predictor of full-time outcomes")
print("2. Goal-related stats are crucial, but other metrics still offer predictive power")
print("3. Machine learning models — especially SVM — can detect meaningful patterns from first-half data")
print("4. Cross-validation confirms the robustness of these findings across data subsets")


# Conclusion

This research has investigated the relationship between the first half and the final result in football matches, through a rigorous methodological approach that combined classical statistical analysis and machine learning models. The Cramer's V index of 0.494 revealed a moderate but significant association, confirmed by the observation that 62.63% of matches end with the same result as the first half.

The implementation of different classifiers, supported by cross-validation techniques and ROC analysis, has allowed us to precisely quantify the predictive capacity of first half statistics. SVM proved to be the most effective model in identifying patterns in partial data, achieving an accuracy of 55% and an AUC of 0.72, significantly higher than random prediction.

The comparative analysis between models with and without goal statistics revealed particularly interesting aspects: while the absence of these metrics reduced the accuracy of the complete match by 25.61%, the impact on first half models was surprisingly contained (only 3.51%). This suggests that although goals are determinant, other statistics such as Expected Goals, shots on target, and duels won contain almost equivalent predictive information.

These results provide an empirical basis for understanding the dynamics of football matches: the first half is not simply a prologue, but contains significant predictive signals about the final outcome. For coaches, analysts, and enthusiasts, this research offers quantitative tools to interpret what happens in the first 45 minutes and how this may influence the final result.

In an era where data analysis is transforming the understanding of football, this study demonstrates that by combining traditional statistics with advanced machine learning techniques, it is possible to reveal hidden patterns that govern the evolution of matches, opening new perspectives for both tactical analysis and strategic decisions.